# FastH3 V2＋新INT8 VAE版

元のFastH3 Colab Notebookを基にした追加版です。**8ステップのV2モデル**と、映像を変換する**INT8 VAE**をセットで使います。
元の4ステップ版とは別Notebookです。元版のランタイムを流用せず、新しいランタイムで開始してください。

- ComfyUI v0.36.0を固定。暫定SolAttnMiniMaxノードと旧専用wheelを、標準BlockSparseAttention＋comfy-kitchen 0.2.34に置換。
- 動画shift 10／音声shift 3／Euler／8ステップ／VSA保持率20%。同梱4ワークフローとアプリ表示にも反映。
- GPUはL4またはA100。モデル合計は約42GB。モデル保存と作業領域には余裕を持たせてください。
- まず0.4MP・短尺で開始。高解像度、長尺、多数の参照は必要メモリと時間が増えます。
- ローカルRTX 4090では1344×768・8秒・参照7枚で成功。**このNotebookのColab L4/A100実機での生成テストは未実施**です。ローカルの結果はColabの速度・品質保証ではありません。
- V2は公式にはT2V（文章から動画）が中心。I2V（開始画像）／R2V（参照画像）は実験的で、画質と演出の一致は別に評価してください。
- 元NotebookのPinggy接続、Colabシークレット、Drive出力、サンプル画像、4種類の操作画面を引き継いでいます。

制作：ざすこ（道草 雑草子）／[AIみちくさch](https://www.youtube.com/channel/UC84fyKjiilxssZVxhE_RiaA)

## このNotebookの保存方式
モデルはDriveに保存し、起動時にColab一時ディスクへコピーします。Driveに約42GB＋余裕が必要です。出力もDrive保存できます。

## 🔗 事前準備：Pinggy Token の設定
このNotebookは、Google Colab上のComfyUIを外部（ブラウザ・PC側ツール等）から使えるように、**Pinggy**というトンネルサービスで一時的な公開URLを発行します。ComfyUIを開くには、この設定が必須です。

> 💡 **Pinggyとは**：Colabのサーバーは通常インターネットから直接アクセスできないため、Pinggyが「Colab内で動いているComfyUI」と「あなたのブラウザ」をつなぐ一時的な公開URL（トンネル）を作ってくれます。このURLは**このColabセッション限りの一時URL**で、Notebookを再起動すると変わります。

**① Pinggyアカウント・トークンの取得**
1. [https://pinggy.io](https://pinggy.io) でアカウントを作成（無料枠あり、Proにすると60分の接続制限が外れます）
2. ダッシュボードから接続トークン（アクセストークン）を取得してコピー

**② Google Colab シークレットへの登録**
1. 左サイドバーの 🔑 **シークレット**（鍵アイコン）を開く
2. **＋ 新しいシークレットを追加** をクリック
3. 名前に `PINGGY_TOKEN`、値にコピーしたトークンを貼り付けて保存
4. **ノートブックからのアクセス** をオンにする

> ⚠️ トークンはシークレットで管理し、コードに直接貼り付けないでください。


# 起動
GPUをL4またはA100に変更し、PINGGY_TOKENをシークレットへ登録して、次のセルを実行してください。
モデルURLはV2＋新VAE用に設定済みです。入力欄はそのまま使えます。


In [ ]:
import base64
# @markdown # 👈この（▶）ボタンをクリックしてComfyUIを起動（☕起動まで約10数分）
# @markdown ### ※停止と再起動もこのボタンで行います。
# @markdown ## 🌱Pinggy接続後、実行ログに表示される青い「ComfyUIを開く」ボタンをクリックしてください。

# ==========================================
# 🔧 PyTorch系ライブラリの軽量チェック（再起動ループ回避版）
# ==========================================
# 重要：このセル内では、現在のColabカーネルに torch / triton を直接 import しません。
# 理由：torchをimportすると内部でtritonも読み込まれ、その後pipがtriton周辺を確認しただけで
#       Colabが「セッションを再起動してください」と表示しやすくなるためです。
# 方針：別プロセスで確認し、必要な場合だけ修復します。
import sys, subprocess

CORE_TORCH_PACKAGES = [
    "torch==2.9.0",
    "torchvision==0.24.0",
    "torchaudio==2.9.0",
]


def run(cmd):
    print("$", " ".join(cmd))
    subprocess.check_call(cmd)


def torch_stack_ok():
    check_code = r'''
import sys
try:
    import torch, torchvision, torchaudio
    print("torch:", torch.__version__)
    print("torchvision:", torchvision.__version__)
    print("torchaudio:", torchaudio.__version__)
    print("torch cuda:", torch.version.cuda)
    print("cuda available:", torch.cuda.is_available())
    ok = (torch.__version__ == "2.9.0+cu130" and torchvision.__version__ == "0.24.0+cu130" and torchaudio.__version__ == "2.9.0+cu130" and torch.cuda.is_available())
    sys.exit(0 if ok else 2)
except Exception as e:
    print("PyTorch check failed:", repr(e))
    sys.exit(1)
'''
    result = subprocess.run([sys.executable, "-c", check_code], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    return result.returncode == 0

if not torch_stack_ok():
    print("=" * 70)
    print("🔧 torch / torchvision / torchaudio を CUDA 13.0版(cu130)にそろえます")
    print("※必要な時だけ修復します。修復後に「セッションを再起動してください」と表示された場合は、")
    print("　その時だけ再起動し、このセルより上には戻らずこのセルから再実行してください。")
    print("=" * 70)
    run([sys.executable, "-m", "pip", "install", "--no-cache-dir", *CORE_TORCH_PACKAGES,
         "--index-url", "https://download.pytorch.org/whl/cu130"])
    if not torch_stack_ok():
        raise RuntimeError("torch / torchvision / torchaudio の修復に失敗しました。ランタイムを新規にして再実行してください。")

print("✅ PyTorch系ライブラリ確認OK。このままComfyUI起動処理に進みます。")

# ==========================================
# 🎮 GPUチェック（A100 または L4 を確認、それ以外は停止）
# ==========================================
# @markdown ### GPUチェック
# @markdown 🌱このNotebookはFastH3 V2＋新INT8 VAE用です。
# @markdown **L4 または A100を選択**してください。この版のColab実機生成は未検証です。
# @markdown それ以外のGPU（T4等）はVRAM的に厳しいため、チェックを外さない限り止めます。
require_a100_or_l4 = True  # @param {type:"boolean"}


def _check_gpu_for_h3(require_a100_or_l4):
    import subprocess
    try:
        gpu_name = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            text=True,
        ).strip().splitlines()[0]
    except Exception as e:
        print(f"⚠️ GPU情報を取得できませんでした: {e}")
        return None
    print(f"🎮 検出されたGPU: {gpu_name}")
    if "L4" in gpu_name:
        print("✅ L4を検出しました。まず低解像度・短尺で確認してください。")
        return "L4"
    if "A100" in gpu_name:
        print("✅ A100を検出しました。このV2版のColab実機性能は未検証です。")
        return "A100"
    print("⚠️ A100/L4ではないGPUです。MiniMax H3（量子化モデルでも約40GB前後）はVRAM不足で失敗する可能性が高いです。")
    if require_a100_or_l4:
        raise RuntimeError(
            "MiniMax H3にはL4 24GB（推奨・コンピューティングユニット節約）またはA100 80GBランタイムが必要です。\n"
            "『ランタイム』→『ランタイムのタイプを変更』でL4かA100を選択し、\n"
            "『ランタイムを接続解除して削除』後にこのセルを再実行してください。\n"
            "（このチェックを飛ばしたい場合は require_a100_or_l4 のチェックを外してください）"
        )
    print("➡️ require_a100_or_l4がオフのため、このまま続行します（自己責任）。")
    return gpu_name

_detected_gpu_for_h3 = _check_gpu_for_h3(require_a100_or_l4)
using_L4_GPU = bool(_detected_gpu_for_h3) and "L4" in str(_detected_gpu_for_h3)
include_manager = False # @param {type:"boolean"}

# ==========================================
# 📁 Google Drive マウント
# ==========================================
# @markdown ## 📁 Google Driveに接続
# @markdown ComfyUIの出力ファイルをバックアップする場合はチェックを入れてください

# @markdown ---
# @markdown ### Google Driveを使用する
use_google_drive = True  # @param {type:"boolean"}

# @markdown ---

if use_google_drive:
    from google.colab import drive
    _drive_mount_ok = False
    for _mount_attempt in range(3):
        try:
            drive.mount('/content/drive', force_remount=(_mount_attempt > 0))
            _drive_mount_ok = True
            break
        except Exception as _mount_err:
            print(f"⚠️ Google Driveのマウントに失敗しました（{_mount_attempt + 1}/3回目）: {_mount_err}")
            print("   Googleの認証ポップアップが出ていたら、許可まで進めてから少し待ってください。")
            import time as _mount_time
            _mount_time.sleep(5)
    if not _drive_mount_ok:
        raise RuntimeError(
            "Google Driveのマウントに3回失敗しました。\n"
            "『ランタイム』→『ランタイムを接続解除して削除』後にもう一度実行するか、\n"
            "ブラウザのポップアップブロック・サードパーティCookie設定を確認してください。"
        )

    print("=" * 70)
    print("✅ Google Driveがマウントされました")
    print("📁 マウント先: /content/drive/MyDrive")
    print("=" * 70)
else:
    print("=" * 70)
    print("⏸️  Google Driveは使用しません")
    print("💡 出力ファイルは/content/ComfyUI/outputに保存されます")
    print("   （ランタイム終了時に消えるので注意してください）")
    print("=" * 70)


# ==========================================
# 📤 Google Drive 出力設定
# ==========================================
# @markdown ## 📤 Google Drive 出力設定
# @markdown ### ComfyUIの出力先をGoogle Driveに変更する
enable_gdrive_output = True  # @param {type:"boolean"}

# @markdown ### 出力先フォルダ設定
GDRIVE_OUTPUT = "/content/drive/MyDrive/ComfyUI_output"  # @param {type:"string"}

# @markdown ---

# 出力フォルダの作成
if use_google_drive and enable_gdrive_output:
    from pathlib import Path
    output_path = Path(GDRIVE_OUTPUT)

    if output_path.exists():
        print(f"📁 既存のフォルダを使用: {GDRIVE_OUTPUT}")
    else:
        output_path.mkdir(parents=True, exist_ok=True)
        print(f"📁 新規フォルダを作成: {GDRIVE_OUTPUT}")

    print("=" * 70)
    print("✅ ComfyUIの出力先がGoogle Driveに設定されます")
    print(f"📁 保存先: {GDRIVE_OUTPUT}")
    print("💡 生成したファイルが直接Google Driveに保存されます")
    print("=" * 70)
elif enable_gdrive_output and not use_google_drive:
    print("=" * 70)
    print("⚠️ Google Drive出力が有効ですが、Google Driveがマウントされていません")
    print("💡 use_google_driveにチェックを入れてください")
    print("=" * 70)
else:
    print("=" * 70)
    print("⏸️  Google Drive出力は無効です")
    print("💡 出力ファイルは/content/ComfyUI/outputに保存されます")
    print("=" * 70)


%cd /content
from IPython.display import clear_output
clear_output()
# torch / triton はColab標準または上のチェック結果を使います。
# xformersだけを --no-deps で入れ、torch/tritonを勝手に入れ替えないようにします。
!python -m pip install -q torchsde einops diffusers accelerate
# cu130 experiment: xformers 0.0.32.post1 is built for cu128, compatibility with cu130 unconfirmed.
# Skipping to avoid conflicts (ComfyUI runs fine without xformers).
# !python -m pip install -q --no-deps xformers==0.0.32.post1
# pipがtorch/triton系を別バージョンへ動かさないための制約ファイル
# Colabの通常Pythonセルでは bash の here-document が壊れやすいため、Pythonで安全に書き出します。
def _installed_version(_name):
    try:
        import importlib.metadata as _md
        return _md.version(_name)
    except Exception:
        return None
_np_ver = _installed_version('numpy')
_pb_ver = _installed_version('protobuf')
_constraints = ['torch==2.9.0', 'torchvision==0.24.0', 'torchaudio==2.9.0', 'triton==3.5.0']
if _np_ver: _constraints.append('numpy==' + _np_ver)
if _pb_ver: _constraints.append('protobuf==' + _pb_ver)
with open('/content/torch_core_constraints.txt', 'w', encoding='utf-8') as f:
    f.write(chr(10).join(_constraints) + chr(10))
print('🔒 バージョン固定:', ' / '.join(_constraints))
!pip install av spandrel albumentations onnx opencv-python onnxruntime -c /content/torch_core_constraints.txt
!pip install color-matcher -c /content/torch_core_constraints.txt
print("ℹ️ protobufはColab既定のまま使います")
!pip install onnxruntime-gpu -c /content/torch_core_constraints.txt
# FastH3 V2: official ComfyUI v0.36.0, pinned to the locally tested revision.
FASTH3_COMFYUI_COMMIT = "ee71d5c4993f29086b27fde1629a945ae48425bf"
from pathlib import Path
import os as _os_cl, shutil as _sh_cl, tarfile as _tarfile, urllib.request as _request
_COMFY = "/content/ComfyUI"
_marker = _os_cl.path.join(_COMFY, "FASTH3_COMMIT.txt")
if _os_cl.path.exists(_os_cl.path.join(_COMFY, "main.py")):
    _head = open(_marker).read().strip() if _os_cl.path.exists(_marker) else ""
    if _head != FASTH3_COMFYUI_COMMIT:
        raise RuntimeError("別バージョンのComfyUIが残っています。『ランタイムを接続解除して削除』後、新しいランタイムで実行してください。Driveのモデルは削除しません。")
else:
    _TMP = "/content/_fasth3_v2_source"
    _TAR = "/content/comfyui_v2.tar.gz"
    _sh_cl.rmtree(_TMP, ignore_errors=True)
    _os_cl.makedirs(_TMP, exist_ok=True)
    _request.urlretrieve(f"https://codeload.github.com/Comfy-Org/ComfyUI/tar.gz/{FASTH3_COMFYUI_COMMIT}", _TAR)
    with _tarfile.open(_TAR) as _tf:
        _tf.extractall(_TMP, filter="data")
    _roots = [p for p in Path(_TMP).iterdir() if (p / "main.py").is_file()]
    if len(_roots) != 1:
        raise RuntimeError("ComfyUIの取得に失敗しました。")
    _os_cl.makedirs(_COMFY, exist_ok=True)
    for _item in _roots[0].iterdir():
        _dst = Path(_COMFY) / _item.name
        if _dst.exists():
            raise RuntimeError(f"既存ファイルと競合しています: {_dst}。新しいランタイムで実行してください。")
        _sh_cl.move(str(_item), str(_dst))
    Path(_marker).write_text(FASTH3_COMFYUI_COMMIT)
    _sh_cl.rmtree(_TMP)
    _os_cl.remove(_TAR)
for _required in ["comfy_extras/nodes_minimax_h3.py", "comfy_extras/nodes_sparse_attention.py"]:
    if not (Path(_COMFY) / _required).is_file():
        raise RuntimeError(f"必要な標準ノードがありません: {_required}")
print("ComfyUI v0.36.0:", FASTH3_COMFYUI_COMMIT)
%cd /content
run([sys.executable, "-m", "pip", "install", "-r", "/content/ComfyUI/requirements.txt", "-c", "/content/torch_core_constraints.txt"])
clear_output()

!mkdir -p /content/ComfyUI/custom_nodes
%cd /content/ComfyUI/custom_nodes
# 🌱 MiniMax H3の公式ワークフローはComfyUIコアだけで完結し、カスタムノードは不要と確認済み
# （ComfyMathExpression含む全ノードがcomfy_extras=コア機能）。起動を速くするため、
# 以前入れていた汎用カスタムノード群（GGUF/VideoHelperSuite/KJNodes/rgthree/essentials/
# LogicUtils/VFI/Frame-Interpolation/PainterI2V）は撤去し、ComfyUI-Managerだけ任意で残す。
if include_manager:
    !git clone https://github.com/ltdrdata/ComfyUI-Manager

# Standard BlockSparseAttention replaces the old PR-only custom node/wheel.
run([sys.executable, "-m", "pip", "install", "comfy-kitchen==0.2.34", "comfy-aimdo==0.5.3", "-c", "/content/torch_core_constraints.txt"])
run([sys.executable, "-c", "import comfy_kitchen as ck; assert callable(getattr(ck, 'sol_attn', None)), 'comfy-kitchen sol_attn unavailable'; print('comfy-kitchen sol_attn: OK')"])
# ③ GPUの確認（Sol-Attnのカーネルは Compute Capability 8.0 以上を要求する）
def _check_gpu_for_fasth3():
    import subprocess
    try:
        name = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
            text=True).strip()
        print("GPU:", name)
        cc = float(name.split(",")[1])
        if cc < 8.0:
            print("=" * 70)
            print("❌ このGPUではFastH3の高速化が効きません（Compute Capability 8.0未満）")
            print("   T4(7.5)は対象外です。ランタイムのタイプを L4 または A100 に変更してください。")
            print("=" * 70)
        else:
            print("✅ Sol-Attn対応のGPUです（Compute Capability", cc, "）")
    except Exception as e:
        print("⚠️ GPUを確認できませんでした:", repr(e))
_check_gpu_for_fasth3()

if include_manager:
    %cd /content/ComfyUI/custom_nodes/ComfyUI-Manager
    !pip install -r requirements.txt -c /content/torch_core_constraints.txt

# 🆕 protobufはColab既定のものをそのまま使う。
# 以前ここで protobuf==5.29.1 を入れ直していたが、このセルが先に書いた制約ファイル
# （Colab既定の 5.29.6 で固定）と衝突して ResolutionImpossible で失敗していた。

clear_output()

%cd /content/ComfyUI

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import subprocess
import sys
from pathlib import Path

def install_apt_packages():
    packages = ['aria2']

    try:
        # Run apt install silently (using -qq)
        subprocess.run(
            ['apt-get', '-y', 'install', '-qq'] + packages,
            check=True,
            capture_output=True
        )
        print("✓ apt packages installed")
    except subprocess.CalledProcessError as e:
        print(f"✗ Error installing apt packages: {e.stderr.decode().strip() or 'Unknown error'}")

print("Installing apt packages...")
install_apt_packages()

def download_with_aria2c(link, folder="/content/ComfyUI/models/loras", hf_token=None):
    import os
    from urllib.parse import urlparse, unquote

    # Hugging Faceなど、URLの末尾に正式なファイル名が入っている通常リンク用
    path_name = os.path.basename(urlparse(link).path)
    filename = unquote(path_name) if path_name else "downloaded_model.safetensors"
    header_arg = ""
    if hf_token and "huggingface.co" in link.lower():
        header_arg = f' --header="Authorization: Bearer {hf_token}"'
    command = f"aria2c --console-log-level=error -c -x 16 -s 16 -k 1M{header_arg} \"{link}\" -d \"{folder}\" -o \"{filename}\""

    print("Executing download command:")
    print(command.replace(hf_token, "***") if hf_token else command)

    os.makedirs(folder, exist_ok=True)
    get_ipython().system(command)

    return filename

def add_civitai_token_to_url(civitai_link, civitai_token=None):
    """CivitaiのURLにtokenを安全に追加する。既にtokenがある場合はそのまま使う。"""
    from urllib.parse import urlparse, parse_qsl, urlencode, urlunparse

    parsed = urlparse(civitai_link)
    if not parsed.scheme or not parsed.netloc:
        raise ValueError("Invalid Civitai URL format. Please use a link like: https://civitai.com/api/download/models/1523247?...")

    query = dict(parse_qsl(parsed.query, keep_blank_values=True))
    if civitai_token and "token" not in query:
        query["token"] = civitai_token

    return urlunparse((parsed.scheme, parsed.netloc, parsed.path, parsed.params, urlencode(query), parsed.fragment))

def pick_new_downloaded_file(folder, before_files):
    """wget実行後に増えたファイルから、実体のある最新ファイルを推定する。"""
    import os

    after_files = set(os.listdir(folder))
    candidates = []
    for name in after_files - before_files:
        path = os.path.join(folder, name)
        if os.path.isfile(path) and os.path.getsize(path) > 0:
            # wgetの一時ファイルやログっぽいものは除外
            if not name.endswith((".tmp", ".aria2")):
                candidates.append(path)

    if not candidates:
        return None

    candidates.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    return candidates[0]

def download_civitai_model(civitai_link, civitai_token, folder="/content/ComfyUI/models/loras"):
    """
    Civitai専用ダウンロード。
    重要：-Oで固定名を付けず、wgetの --content-disposition / --trust-server-names を使う。
    これにより、Civitaiが返す正式ファイル名（例：FlowCamera_epoch67.safetensors）で保存する。
    """
    import os
    import subprocess

    os.makedirs(folder, exist_ok=True)
    civitai_url = add_civitai_token_to_url(civitai_link, civitai_token)

    before_files = set(os.listdir(folder))

    cmd = [
        "wget",
        "--max-redirect=10",
        "--content-disposition",
        "--trust-server-names",
        "--show-progress",
        "-P", folder,
        civitai_url,
    ]

    print("Downloading from Civitai with official filename...")
    print("$ " + " ".join([f'\"{c}\"' if " " in c or "&" in c or "?" in c else c for c in cmd]))

    result = subprocess.run(cmd, text=True)
    if result.returncode != 0:
        print("❌ Civitai download failed.")
        return False

    downloaded_path = pick_new_downloaded_file(folder, before_files)

    # 同名ファイルが既に存在していて新規差分が取れない場合の保険：フォルダ内の最新safetensorsを表示
    if downloaded_path is None:
        safetensors = [
            os.path.join(folder, f) for f in os.listdir(folder)
            if f.lower().endswith((".safetensors", ".ckpt", ".pt", ".pth", ".bin")) and os.path.isfile(os.path.join(folder, f))
        ]
        if safetensors:
            downloaded_path = max(safetensors, key=os.path.getmtime)

    if downloaded_path and os.path.exists(downloaded_path) and os.path.getsize(downloaded_path) > 0:
        filename = os.path.basename(downloaded_path)
        print(f"✅ Civitai model downloaded successfully: {downloaded_path}")
        print(f"📛 Saved filename: {filename}")
        return filename

    print("❌ Civitai download finished, but the saved file could not be detected.")
    return False

def download_lora(link, folder="/content/ComfyUI/models/loras", civitai_token=None):
    """
    Download a model file, automatically detecting if it's a Civitai link or huggingface download.

    Args:
        link: The download URL (either huggingface or Civitai)
        folder: Destination folder for the download
        civitai_token: Optional token for Civitai downloads (required if link is from Civitai)

    Returns:
        The filename of the downloaded model
    """
    if "civitai.com" in link.lower():
        if not civitai_token:
            print("⚠️ Civitaiトークンが未設定のため、トークンなしでダウンロードを試します。")
            print("   非公開・年齢制限・ログイン必須モデルの場合は失敗することがあります。")
        return download_civitai_model(link, civitai_token, folder)
    else:
        return download_with_aria2c(link, folder)

def model_download(url: str, dest_dir: str, filename: str = None, silent: bool = True, hf_token: str = None) -> bool:
    """
    Colab-optimized download with aria2c

    Args:
        url: Download URL
        dest_dir: Target directory (will be created if needed)
        filename: Optional output filename (defaults to URL filename)
        silent: If True, suppresses all output (except errors)
        hf_token: Optional Hugging Face token. Added as an Authorization header
            only when the URL host is huggingface.co (rate-limit avoidance).

    Returns:
        bool: True if successful, False if failed
    """
    try:
        # Create destination directory
        Path(dest_dir).mkdir(parents=True, exist_ok=True)

        # Set filename if not specified
        if filename is None:
            filename = url.split('/')[-1].split('?')[0]  # Remove URL parameters

        # Build command
        cmd = [
            'aria2c',
            '--console-log-level=error',
            '-c', '-x', '16', '-s', '16', '-k', '1M',
            '-d', dest_dir,
            '-o', filename,
            url
        ]
        if hf_token and "huggingface.co" in url.lower():
            cmd.insert(1, f'--header=Authorization: Bearer {hf_token}')

        # Add silent flags if requested
        if silent:
            cmd.extend(['--summary-interval=0', '--quiet'])
            print(f"Downloading {filename}...", end=' ', flush=True)

        # Run download
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)

        if silent:
            print("Done!")
        else:
            print(f"Downloaded {filename} to {dest_dir}")
        return filename

    except subprocess.CalledProcessError as e:
        error = e.stderr.strip() or "Unknown error"
        print(f"\nError downloading {filename}: {error}")
        return False
    except Exception as e:
        print(f"\nError: {str(e)}")
        return False

# ========================================
# ダウンロード処理関数
# ========================================

def parse_urls(url_string):
    """
    複数行のURL文字列をパースして、有効なURLのリストを返す

    Args:
        url_string: 改行区切りのURL文字列

    Returns:
        list: 有効なURLのリスト
    """
    urls = []
    for line in url_string.strip().split('\n'):
        line = line.strip()
        # コメント行（#始まり）と空行をスキップ
        if line and not line.startswith('#') and line.startswith('http'):
            urls.append(line)
    return urls

# ========================================
# 🌱💾 Google Driveへのモデル永続保存
# ========================================
# H3のモデル本体（合計約40〜60GB）を、毎回Colabへダウンロードし直すのではなく
# Google Driveの下記フォルダに保存し、2回目以降はそれを再利用する。
# 初回だけダウンロードが発生し、2回目以降はセットアップがぐっと速くなる。

# @markdown ---
# @markdown ## 💾 Google Driveへのモデル永続保存
# @markdown モデル本体をGoogle Driveの以下のフォルダに保存し、次回以降は再ダウンロードをスキップします。
DRIVE_MODEL_ROOT = "/content/drive/MyDrive/MiniMaxH3_Models"  # @param {type:"string"}
# @markdown 容量が気になる方はフォルダパスを変更してください（フォルダは自動作成されます）。
# @markdown チェックを外すと、Driveに保存せず毎回ローカルへダウンロードする従来動作になります（Colab終了で消えます）。
persist_models_to_drive = True  # @param {type:"boolean"}

# @markdown ---
# @markdown ## 💾 モデル読み込み方式（既定: ステージングなし・L4実測で安定）
# @markdown Driveのモデルファイルへ直接アクセスします（ローカルへのコピー待ち時間なし）。基本的にONのままで問題ありません。チェックを外すと、従来どおり一度ローカルディスクへコピーしてから使います。
skip_local_staging = True  # @param {type:"boolean"}

import os
import shutil
from pathlib import Path

# 既知のH3公式ファイルは、想定サイズ(GiB)を下回っていたら壊れているとみなして再ダウンロードする
MODEL_MIN_GIB_HINTS = {
    "minimax_h3_ref2va_pruned_int8_convrot.safetensors": 19.0,
    "minimax_h3_fl2va_pruned_int8_convrot.safetensors": 19.0,
    "minimax_h3_ref2va_int8_convrot.safetensors": 19.0,
    "minimax_h3_fl2va_int8_convrot.safetensors": 19.0,
    "fastvideo_fasth3_8step_v2_pruned_int8_convrot.safetensors": 20.5,
    "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors": 14.0,
    "qwen3vl_32b_minimax_h3_int8_convrot.safetensors": 30.0,
    "minimax_h3_video_vae_int8_convrot.safetensors": 2.6,
    "minimax_h3_audio_vae_fp32.safetensors": 0.5,
}


def ensure_model_persistent(url, comfy_folder, filename=None, min_expected_gib=None, civitai_token=None, hf_token=None):
    """
    Google Driveに保存したモデルを再利用しつつ、必要な時だけダウンロードする。
    さらに、実行時の読み込み速度のためColabのローカルディスクへステージング（コピー）してから使う。

    1. Drive上に既に十分なサイズのファイルがあれば、再ダウンロードしない。
       無ければ、まずColabのローカルディスクへダウンロードし、サイズを確認してからDriveへ移動する。
    2. ComfyUIのmodelsフォルダへは、Driveのファイルを直接シンボリックリンクするのではなく、
       いったんColabのローカルディスク（/content/h3_model_stage/）へコピーしてから、
       そのローカルコピーへシンボリックリンクを張る。
       Google Drive越しの読み込み（FUSEマウント）はネットワーク経由になり遅くなりがちなので、
       実際にComfyUIが読みに行く先はローカルSSD相当にすることで高速化する。
       ローカルの空き容量が足りない場合は、従来通りDriveへ直接シンボリックリンクする。
    """
    from urllib.parse import urlparse, unquote

    comfy_dest_dir = Path(comfy_folder)
    comfy_dest_dir.mkdir(parents=True, exist_ok=True)

    is_civitai = "civitai.com" in url.lower()

    if not filename and not is_civitai:
        path_name = os.path.basename(urlparse(url).path)
        filename = unquote(path_name) if path_name else "model.safetensors"

    if not persist_models_to_drive or is_civitai:
        # Drive保存なし（従来通りローカルへ直接ダウンロード）。
        # CivitaiはダウンロードするまでファイルGname確定しないため、Drive方式の対象外にしている。
        if is_civitai:
            download_lora(url, folder=str(comfy_dest_dir), civitai_token=civitai_token)
        else:
            model_download(url, str(comfy_dest_dir), filename=filename)
        return

    if min_expected_gib is None:
        min_expected_gib = MODEL_MIN_GIB_HINTS.get(filename)

    drive_dir = Path(DRIVE_MODEL_ROOT)
    drive_dir.mkdir(parents=True, exist_ok=True)
    drive_path = drive_dir / filename
    comfy_path = comfy_dest_dir / filename

    min_expected_bytes = int(min_expected_gib * (1024 ** 3) * 0.98) if min_expected_gib else 1

    if drive_path.is_file() and drive_path.stat().st_size >= min_expected_bytes:
        print(f"✅ Driveに保存済みのモデルを再利用: {filename}（{drive_path.stat().st_size / 1024**3:.2f} GiB）")
    else:
        print(f"⬇️ Driveに未保存のため新規ダウンロード: {filename}")
        tmp_dir = Path("/content/h3_model_tmp")
        tmp_dir.mkdir(parents=True, exist_ok=True)
        import time as _dl_time_mod
        _dl_started = _dl_time_mod.time()
        tmp_filename = model_download(url, str(tmp_dir), filename=filename, hf_token=hf_token)
        _dl_elapsed = _dl_time_mod.time() - _dl_started
        tmp_path = tmp_dir / (tmp_filename or filename)
        if not tmp_path.is_file():
            print(f"❌ ダウンロードに失敗しました: {filename}")
            return
        if min_expected_gib and tmp_path.stat().st_size < min_expected_bytes:
            print(f"⚠️ ファイルサイズが想定より小さいです（{tmp_path.stat().st_size / 1024**3:.2f} GiB）。破損の可能性があるため保存を中止します。")
            return
        _dl_mbps = (tmp_path.stat().st_size / 1024 / 1024) / _dl_elapsed if _dl_elapsed > 0 else 0
        print(f"⬇️ ダウンロード完了: {filename}（{_dl_elapsed:.1f}秒、{_dl_mbps:.1f} MB/s、aria2c 16並列）")
        print(f"📦 Google Driveへ移動中...（{tmp_path.stat().st_size / 1024**3:.2f} GiB、少し時間がかかります）")
        if drive_path.exists():
            drive_path.unlink()
        shutil.move(str(tmp_path), str(drive_path))
        print(f"✅ Driveへ保存しました: {drive_path}")

    # ── ここからローカルステージング ──
    stage_dir = Path("/content/h3_model_stage")
    stage_dir.mkdir(parents=True, exist_ok=True)
    stage_path = stage_dir / filename
    drive_size = drive_path.stat().st_size

    link_target = drive_path  # 既定はDriveへ直接（空き容量不足などの保険）

    stage_ok = stage_path.is_file() and not stage_path.is_symlink() and stage_path.stat().st_size == drive_size
    if skip_local_staging:
        print(f"🧪 実験: ローカルステージングをスキップし、Driveへ直接シンボリックリンクします: {filename}")
        link_target = drive_path
    elif stage_ok:
        print(f"✅ ローカルステージング済みを再利用: {filename}")
        link_target = stage_path
    else:
        free_gib = shutil.disk_usage("/content").free / (1024 ** 3)
        needed_gib = drive_size / (1024 ** 3) + 5  # 5GiBの余裕を見る
        if free_gib < needed_gib:
            print(f"⚠️ ローカルディスクの空きが足りないため（空き{free_gib:.1f}GiB／必要{needed_gib:.1f}GiB）、Driveから直接読み込みます: {filename}")
        else:
            print(f"📥 ローカルディスクへステージング中...（{drive_size / 1024**3:.2f} GiB、少し時間がかかります）")
            stage_tmp = stage_path.with_suffix(stage_path.suffix + ".copying")
            import time as _stage_time_mod
            _stage_started = _stage_time_mod.time()
            with open(drive_path, "rb") as _src, open(stage_tmp, "wb") as _dst:
                shutil.copyfileobj(_src, _dst, length=64 * 1024 * 1024)
            _stage_elapsed = _stage_time_mod.time() - _stage_started
            _stage_mbps = (drive_size / 1024 / 1024) / _stage_elapsed if _stage_elapsed > 0 else 0
            stage_tmp.replace(stage_path)
            print(f"✅ ローカルステージング完了: {filename}（{_stage_elapsed:.1f}秒、{_stage_mbps:.1f} MB/s）")
            link_target = stage_path

    if comfy_path.is_symlink():
        if comfy_path.resolve() != link_target.resolve():
            comfy_path.unlink()
            comfy_path.symlink_to(link_target)
    elif comfy_path.exists():
        print(f"⚠️ {comfy_path} は既にシンボリックリンク以外のファイルとして存在するため、そのままにします。")
    else:
        comfy_path.symlink_to(link_target)


def download_models_to_folder(url_string, folder_path, civitai_token=None):
    """
    指定されたフォルダに複数のモデルをダウンロード（Drive永続化対応版）。

    Args:
        url_string: 改行区切りのURL文字列
        folder_path: ダウンロード先フォルダパス
        civitai_token: Civitai APIトークン（オプション）
    """
    urls = parse_urls(url_string)

    if not urls:
        return

    print(f"\n📂 対象フォルダ: {folder_path}")
    print(f"   ファイル数: {len(urls)}")

    for i, url in enumerate(urls, 1):
        if "civitai.com" in url.lower():
            display_name = "Civitai公式ファイル名（ダウンロード後に確定）"
            hint_key = None
        else:
            display_name = url.split('/')[-1].split('?')[0]
            hint_key = display_name
        print(f"   [{i}/{len(urls)}] {display_name}")
        ensure_model_persistent(
            url,
            folder_path,
            filename=None if "civitai.com" in url.lower() else display_name,
            min_expected_gib=MODEL_MIN_GIB_HINTS.get(hint_key) if hint_key else None,
            civitai_token=civitai_token,
        )

    print(f"✓ 完了: {folder_path}\n")


# ========================================
# 📥 各種モデルのダウンロード設定
# ========================================

# @markdown # 📥 モデルダウンロード設定
# @markdown 各フォルダごとに、使用したいモデルのダウンロードURLを入力してください。
# @markdown 複数のURLは改行で区切って入力できます。

# @markdown ---
# @markdown ## 🔑 Civitai API Token（任意・H3本体では不要）
# @markdown H3のモデル本体はHugging Faceから取得するため、この設定はH3を使うだけなら不要です。Civitai経由で追加のカスタムモデルをDLしたい場合のみ、`CIVITAI_KEY`という名前でColabの🔑シークレットに登録してください。

civitai_token = ""

try:
    from google.colab import userdata
    civitai_token = userdata.get('CIVITAI_KEY')
    if civitai_token:
        civitai_token = civitai_token.strip()
        print("✅ Civitai API キーをシークレットから読み込みました")
except Exception:
    pass

if not civitai_token:
    print("ℹ️ Civitai APIキー未設定です（H3の利用には不要なので、そのままで問題ありません）")

# @markdown ---
# @markdown ## 🔑 Hugging Face API Token（任意・推奨）
# @markdown H3のモデル本体はHugging Faceの公開リポジトリ（`Comfy-Org/MiniMax-H3`）にあり、トークンが無くてもダウンロードできます。
# @markdown ただし、トークンを登録しておくとダウンロード時のレート制限（アクセス制限）にかかりにくくなるため、設定を推奨します。
# @markdown
# @markdown **① トークンの取得**
# @markdown 1. [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) を開く（要ログイン）
# @markdown 2. **＋ Create new token** をクリック（種類は「Read」で十分）
# @markdown 3. 名前を入力して作成 → 表示されたトークンをコピー
# @markdown
# @markdown **② Google Colab シークレットへの登録**
# @markdown 1. 左サイドバーの 🔑 **シークレット**（鍵アイコン）を開く
# @markdown 2. **＋ 新しいシークレットを追加** をクリック
# @markdown 3. 名前に `HF_TOKEN`、値にコピーしたトークンを貼り付けて保存
# @markdown 4. **ノートブックからのアクセス** をオンにする

hf_token = ""

try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        hf_token = hf_token.strip()
        print("✅ Hugging Face トークンをシークレットから正常に読み込みました")
    else:
        print("ℹ️ 'HF_TOKEN' は見つかりましたが、中身が空っぽのようです（公開モデルなのでトークンなしでも続行できます）")
except Exception as e:
    print(f"ℹ️ Hugging Face トークンの読み込みをスキップします: {e}")
    print("   （`Comfy-Org/MiniMax-H3` は公開リポジトリのため、トークンが無くてもダウンロード自体は可能です）")
    hf_token = ""

if not hf_token:
    print("ℹ️ Hugging Faceトークンなしで続行します。レート制限にかかった場合は上記手順でHF_TOKENを設定してください。")

# @markdown ---
# @markdown ## 📁 diffusion_models フォルダ
# @markdown 🌱FastH3 V2（8ステップ）本体。I2V/R2Vは実験的です。新VAEと以下の専用ワークフローを使用します。
diffusion_models_url_1 = "https://huggingface.co/FastVideo/FastVideo-FastH3-Comfy/resolve/ec1e3aa374a91c57b0b94a1623b7e657c0498cf2/diffusion_models/fastvideo_fasth3_8step_v2_pruned_int8_convrot.safetensors" # @param {type:"string"}
diffusion_models_url_2 = "" # @param {type:"string"}
diffusion_models_url_3 = "" # @param {type:"string"}
diffusion_models_url_4 = "" # @param {type:"string"}
diffusion_models_url_5 = "" # @param {type:"string"}
diffusion_models_url_6 = "" # @param {type:"string"}
diffusion_models_url_7 = "" # @param {type:"string"}
diffusion_models_url_8 = "" # @param {type:"string"}

# 入力されたURLを結合
diffusion_models_urls = "\n".join([
    url for url in [
        diffusion_models_url_1,
        diffusion_models_url_2,
        diffusion_models_url_3,
        diffusion_models_url_4,
        diffusion_models_url_5,
        diffusion_models_url_6,
        diffusion_models_url_7,
        diffusion_models_url_8
    ] if url.strip()
])

# @markdown ---
# @markdown ## 📁 text_encoders フォルダ
# @markdown MiniMax H3用 テキストエンコーダー（Qwen3-VL-32B 量子化NVFP4-AWQ版・約14.6GiB）。プロンプトと参照画像・参照動画の理解に使用
text_encoders_url_1 = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/7a2065e37f5ff9d3c4e605f164d4cac388eff8e8/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors" # @param {type:"string"}
text_encoders_url_2 = "" # @param {type:"string"}
text_encoders_url_3 = "" # @param {type:"string"}

# 入力されたURLを結合
text_encoders_urls = "\n".join([url for url in [text_encoders_url_1, text_encoders_url_2, text_encoders_url_3] if url.strip()])

# @markdown ---
# @markdown ## 📁 vae フォルダ
# @markdown MiniMax H3用 動画VAEと音声VAE（H3は動画と音声を同時に扱うため両方必須）
vae_url_1 = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/7a2065e37f5ff9d3c4e605f164d4cac388eff8e8/vae/minimax_h3_video_vae_int8_convrot.safetensors" # @param {type:"string"}
vae_url_2 = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/7a2065e37f5ff9d3c4e605f164d4cac388eff8e8/vae/minimax_h3_audio_vae_fp32.safetensors" # @param {type:"string"}
vae_url_3 = "" # @param {type:"string"}

# 入力されたURLを結合
vae_urls = "\n".join([url for url in [vae_url_1, vae_url_2, vae_url_3] if url.strip()])

# @markdown ---
# @markdown ## 📁 clip_vision フォルダ
# @markdown CLIP Visionモデル（画像理解用）
clip_vision_url_1 = "" # @param {type:"string"}
clip_vision_url_2 = "" # @param {type:"string"}
clip_vision_url_3 = "" # @param {type:"string"}

# 入力されたURLを結合
clip_vision_urls = "\n".join([url for url in [clip_vision_url_1, clip_vision_url_2, clip_vision_url_3] if url.strip()])

# @markdown ---
# @markdown ## 📁 loras フォルダ
# @markdown MiniMax H3用の公式高速化LoRA（Lightning/Turbo等）は2026-08-03時点で確認されていないため既定は空欄。必要なLoRAが見つかった場合のみ記入する

loras_url_1 = "" # @param {type:"string"}
loras_url_2 = "" # @param {type:"string"}
loras_url_3 = "" # @param {type:"string"}
loras_url_4 = "" # @param {type:"string"}
loras_url_5 = "" # @param {type:"string"}
loras_url_6 = "" # @param {type:"string"}
loras_url_7 = "" # @param {type:"string"}
loras_url_8 = "" # @param {type:"string"}

# URLをまとめる
loras_urls = "\n".join([url for url in [loras_url_1, loras_url_2, loras_url_3, loras_url_4, loras_url_5, loras_url_6, loras_url_7, loras_url_8] if url])

# @markdown ---
# @markdown ## 📁 audio_encoders フォルダ
# @markdown Audio Encoderモデル（音声付き動画生成用）
audio_encoders_url_1 = "" # @param {type:"string"}
audio_encoders_url_2 = "" # @param {type:"string"}
audio_encoders_url_3 = "" # @param {type:"string"}

# URLをまとめる
audio_encoders_urls = "\n".join([url for url in [audio_encoders_url_1, audio_encoders_url_2, audio_encoders_url_3] if url])

# @markdown ---
# @markdown ## 📁 sam2 フォルダ
# @markdown SAM2セグメンテーションモデル
sam2_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 unet フォルダ
# @markdown UNetモデル（Flux等）
unet_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 checkpoints フォルダ
# @markdown チェックポイントモデル
checkpoints_url_1 = "" # @param {type:"string"}
checkpoints_url_2 = "" # @param {type:"string"}
checkpoints_url_3 = "" # @param {type:"string"}
checkpoints_url_4 = "" # @param {type:"string"}
checkpoints_url_5 = "" # @param {type:"string"}

# 入力されたURLを結合
checkpoints_urls = "\n".join([url for url in [checkpoints_url_1, checkpoints_url_2, checkpoints_url_3, checkpoints_url_4, checkpoints_url_5] if url.strip()])

# @markdown ---
# @markdown ## 📁 controlnet フォルダ
# @markdown ControlNetモデル
controlnet_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 clip フォルダ
# @markdown CLIPモデル
clip_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 upscale_models フォルダ
# @markdown アップスケールモデル
upscale_models_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 embeddings フォルダ
# @markdown Embeddingsモデル
embeddings_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 カスタムフォルダ（自由指定）
# @markdown フォルダパスとURLを指定
custom_folder_path = "" # @param {type:"string"}
custom_folder_urls = "" # @param {type:"string"}


# ========================================
# ダウンロード実行
# ========================================

print("=" * 70)
print("🚀 Starting model downloads...")
print("=" * 70)

# 各フォルダへのダウンロード
download_models_to_folder(diffusion_models_urls, "/content/ComfyUI/models/diffusion_models", civitai_token)
download_models_to_folder(text_encoders_urls, "/content/ComfyUI/models/text_encoders", civitai_token)
download_models_to_folder(vae_urls, "/content/ComfyUI/models/vae", civitai_token)
download_models_to_folder(clip_vision_urls, "/content/ComfyUI/models/clip_vision", civitai_token)
download_models_to_folder(loras_urls, "/content/ComfyUI/models/loras", civitai_token)
download_models_to_folder(audio_encoders_urls, "/content/ComfyUI/models/audio_encoders", civitai_token)
download_models_to_folder(sam2_urls, "/content/ComfyUI/models/sam2", civitai_token)
download_models_to_folder(unet_urls, "/content/ComfyUI/models/unet", civitai_token)
download_models_to_folder(checkpoints_urls, "/content/ComfyUI/models/checkpoints", civitai_token)
download_models_to_folder(controlnet_urls, "/content/ComfyUI/models/controlnet", civitai_token)
download_models_to_folder(clip_urls, "/content/ComfyUI/models/clip", civitai_token)
download_models_to_folder(upscale_models_urls, "/content/ComfyUI/models/upscale_models", civitai_token)
download_models_to_folder(embeddings_urls, "/content/ComfyUI/models/embeddings", civitai_token)

# カスタムフォルダ
if custom_folder_path and custom_folder_urls.strip():
    download_models_to_folder(custom_folder_urls, custom_folder_path, civitai_token)

print("=" * 70)
print("✓ All downloads completed!")
print("=" * 70)

# ========================================
# 🌱 FastH3用ワークフローの配置
# ========================================
# 20ステップ版のワークフローテンプレート生成（Sol-Attn+EasyCache版の自動生成を含む）は、
# FastH3では設定が噛み合わないため丸ごと差し替えています。
#
# 【重要】配布元のサンプル fasth3_vsa_sample.json は「API形式」で、ノードの座標も
# 接続線の情報も持たないため、ComfyUIの画面で開いても真っ白になります。
# ここでは、それを「UI形式」に変換して動作確認済みのものを埋め込んでいます。
# （2026-09-01にローカルのComfyUIで、実際に画面から開けることを目視確認済み）

import os as _os, json as _json, base64 as _b64

h3_workflow_dir = "/content/ComfyUI/user/default/workflows/FastH3_V2_newVAE"
_os.makedirs(h3_workflow_dir, exist_ok=True)

_WF_T2V_B64 = (
    "eyJpZCI6ImZhc3RoMy10MnYiLCJyZXZpc2lvbiI6MCwibGFzdF9ub2RlX2lkIjoxNiwibGFzdF9saW5rX2lkIjoxOCwibm9kZXMi"
    "Olt7ImlkIjoxLCJ0eXBlIjoiVU5FVExvYWRlciIsInBvcyI6WzAsMF0sInNpemUiOls0MDAsODJdLCJmbGFncyI6e30sIm9yZGVy"
    "IjowLCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiTU9ERUwiLCJ0eXBlIjoiTU9ERUwiLCJsaW5rcyI6"
    "WzFdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJVTkVUTG9hZGVyIn0sIndpZGdldHNfdmFsdWVzIjpbImZh"
    "c3R2aWRlb19mYXN0aDNfOHN0ZXBfdjJfcHJ1bmVkX2ludDhfY29udnJvdC5zYWZldGVuc29ycyIsImRlZmF1bHQiXX0seyJpZCI6"
    "MiwidHlwZSI6Ik1pbmlNYXhIM1NpZ21hU2hpZnQiLCJwb3MiOls0NjAsMF0sInNpemUiOls0MDAsMTA4XSwiZmxhZ3MiOnt9LCJv"
    "cmRlciI6NywibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoibW9kZWwiLCJ0eXBlIjoiTU9ERUwiLCJsaW5rIjoxfV0sIm91dHB1"
    "dHMiOlt7Im5hbWUiOiJNT0RFTCIsInR5cGUiOiJNT0RFTCIsImxpbmtzIjpbMl19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUg"
    "Zm9yIFMmUiI6Ik1pbmlNYXhIM1NpZ21hU2hpZnQifSwid2lkZ2V0c192YWx1ZXMiOlsxMC4wLDMuMF19LHsiaWQiOjMsInR5cGUi"
    "OiJCbG9ja1NwYXJzZUF0dGVudGlvbiIsInBvcyI6WzkyMCwwXSwic2l6ZSI6WzQwMCwzNTBdLCJmbGFncyI6e30sIm9yZGVyIjo5"
    "LCJtb2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJtb2RlbCIsInR5cGUiOiJNT0RFTCIsImxpbmsiOjJ9XSwib3V0cHV0cyI6W3si"
    "bmFtZSI6Ik1PREVMIiwidHlwZSI6Ik1PREVMIiwibGlua3MiOls1XX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZS"
    "IjoiQmxvY2tTcGFyc2VBdHRlbnRpb24ifSwid2lkZ2V0c192YWx1ZXMiOlsidnNhIiwyMCwwLDEsIiIsMCwwLCJleGFjdF9rdl9h"
    "bmRfcm93cyIsdHJ1ZV0sIndpZGdldHNfdmFsdWVzX25hbWVkIjp7InNlbGVjdGlvbi5rZWVwX3BlcmNlbnQiOjIwLCJlbmRfcGVy"
    "Y2VudCI6MSwiZXh0cmFfdG9rZW5zIjowLCJzdGFydF9wZXJjZW50IjowLCJkZW5zZV9ibG9ja3MiOiIiLCJzZWxlY3Rpb24iOiJ2"
    "c2EiLCJ2ZXJib3NlIjp0cnVlLCJzaW5rX2NvbmRpdGlvbmluZyI6ImV4YWN0X2t2X2FuZF9yb3dzIiwibWluX3Rva2VucyI6MH19"
    "LHsiaWQiOjQsInR5cGUiOiJDTElQTG9hZGVyIiwicG9zIjpbMCwyNjBdLCJzaXplIjpbNDAwLDEwOF0sImZsYWdzIjp7fSwib3Jk"
    "ZXIiOjEsIm1vZGUiOjAsImlucHV0cyI6W10sIm91dHB1dHMiOlt7Im5hbWUiOiJDTElQIiwidHlwZSI6IkNMSVAiLCJsaW5rcyI6"
    "WzNdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJDTElQTG9hZGVyIn0sIndpZGdldHNfdmFsdWVzIjpbInF3"
    "ZW4zdmxfMzJiX21pbmltYXhfaDNfbnZmcDRfYXdxLnNhZmV0ZW5zb3JzIiwibWluaW1heCIsImRlZmF1bHQiXX0seyJpZCI6NSwi"
    "dHlwZSI6IlZBRUxvYWRlciIsInBvcyI6WzAsNTIwXSwic2l6ZSI6WzQwMCw2MF0sImZsYWdzIjp7fSwib3JkZXIiOjIsIm1vZGUi"
    "OjAsImlucHV0cyI6W10sIm91dHB1dHMiOlt7Im5hbWUiOiJWQUUiLCJ0eXBlIjoiVkFFIiwibGlua3MiOls0LDEzXX1dLCJwcm9w"
    "ZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiVkFFTG9hZGVyIn0sIndpZGdldHNfdmFsdWVzIjpbIm1pbmltYXhfaDNfdmlk"
    "ZW9fdmFlX2ludDhfY29udnJvdC5zYWZldGVuc29ycyJdfSx7ImlkIjo2LCJ0eXBlIjoiVkFFTG9hZGVyIiwicG9zIjpbMCw3ODBd"
    "LCJzaXplIjpbNDAwLDYwXSwiZmxhZ3MiOnt9LCJvcmRlciI6MywibW9kZSI6MCwiaW5wdXRzIjpbXSwib3V0cHV0cyI6W3sibmFt"
    "ZSI6IlZBRSIsInR5cGUiOiJWQUUiLCJsaW5rcyI6WzE1XX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiVkFF"
    "TG9hZGVyIn0sIndpZGdldHNfdmFsdWVzIjpbIm1pbmltYXhfaDNfYXVkaW9fdmFlX2ZwMzIuc2FmZXRlbnNvcnMiXX0seyJpZCI6"
    "NywidHlwZSI6Ik1pbmlNYXhIM0ltYWdlVG9WaWRlbyIsInBvcyI6WzQ2MCwyNjBdLCJzaXplIjpbNDAwLDIzOF0sImZsYWdzIjp7"
    "fSwib3JkZXIiOjgsIm1vZGUiOjAsImlucHV0cyI6W3sibmFtZSI6ImNsaXAiLCJ0eXBlIjoiQ0xJUCIsImxpbmsiOjN9LHsibmFt"
    "ZSI6InZhZSIsInR5cGUiOiJWQUUiLCJsaW5rIjo0fSx7Im5hbWUiOiJmaXJzdF9mcmFtZSIsInR5cGUiOiJJTUFHRSIsImxpbmsi"
    "Om51bGwsInNoYXBlIjo3fSx7Im5hbWUiOiJsYXN0X2ZyYW1lIiwidHlwZSI6IklNQUdFIiwibGluayI6bnVsbCwic2hhcGUiOjd9"
    "XSwib3V0cHV0cyI6W3sibmFtZSI6InBvc2l0aXZlIiwidHlwZSI6IkNPTkRJVElPTklORyIsImxpbmtzIjpbNl19LHsibmFtZSI6"
    "IkxBVEVOVCIsInR5cGUiOiJMQVRFTlQiLCJsaW5rcyI6WzExXX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoi"
    "TWluaU1heEgzSW1hZ2VUb1ZpZGVvIn0sIndpZGdldHNfdmFsdWVzIjpbImludGVncmF0ZWRfbXVsdGltb2RhbF9kZXNjcmlwdGlv"
    "bjogW1Nob3QgMV0gMkQtYW5pbWF0ZWQsIGFuaW1lIHN0eWxlLCBhIG1lZGl1bSBjbG9zZS11cCBzaG90IGZyYW1lcyBhIGNoZWVy"
    "ZnVsIHlvdW5nIHdvbWFuIHdpdGggZGFyayBoYWlyIHRpZWQgdXAgaW4gYSBuZWF0IGJ1biwgc2lkZSBiYW5ncyBmcmFtaW5nIGhl"
    "ciBkZWxpY2F0ZSBmYWNlLCBhbmQgZGFyayBwdXJwbGUgZXllcyB3ZWFyaW5nIHNtYWxsIHBlYXJsIGVhcnJpbmdzLiBTaGUgd2Vh"
    "cnMgYSBsaWdodCBncmF5IHR1cnRsZW5lY2sga25pdCB0b3AgdW5kZXIgYSBibGFjayBzcGFnaGV0dGktc3RyYXAgbWluaSBkcmVz"
    "cy4gU3RhbmRpbmcgaW4gYSBjbGVhbiBtb2Rlcm4gcm9vbSB3aXRoIHdoaXRlIHdhbGxzIGFuZCBhIGRhcmsgZG9vcmZyYW1lLCBz"
    "aGUgdGlsdHMgaGVyIGhlYWQgc2xpZ2h0bHkgd2l0aCBhIGJyaWdodCBzbWlsZSwgbGlnaHRseSByZXN0aW5nIGhlciByaWdodCBm"
    "aW5nZXJ0aXBzIG5lYXIgaGVyIGNvbGxhcmJvbmUuIFRoZSBjYW1lcmEgcHVzaGVzIGluIHdpdGggc21hbGwgYW1wbGl0dWRlIGF0"
    "IHNsb3cgc3BlZWQgYXMgdGhlIHlvdW5nIHdvbWFuIHdpdGggYSBzd2VldCwgY2xlYXIgdm9pY2UgKFMxKSBnZW50bHkgc2F5czog"
    "PGQ+W0phcGFuZXNlXSDjgZPjgpPjgavjgaHjga/vvIHku4rml6XjgoLkuIDml6XpoJHlvLXjgo3jgYbjga3jgII8L2Q+XG5cbm92"
    "ZXJhbGxfc291bmRzY2FwZTogUXVpZXQgaW5kb29yIHJvb20gdG9uZSwgYSBzb2Z0IHJ1c3RsZSBvZiBrbml0IGZhYnJpYyBhcyBz"
    "aGUgbW92ZXMgaGVyIHNob3VsZGVyLCBhbmQgdGhlIGNsZWFyIHNwb2tlbiB2b2ljZSBpbiBhIG5hdHVyYWwgcm9vbSBhY291c3Rp"
    "Yy5cblxubm9uX2RpZWdldGljX211c2ljOiBBIGdlbnRsZSwgdXBsaWZ0aW5nIGFjb3VzdGljIGd1aXRhciBwYXR0ZXJuIHdpdGgg"
    "d2FybSBwaWFubyBub3RlcyBhdCBhIG1vZGVyYXRlIHRlbXBvLCBmYWRpbmcgc29mdGx5IGF0IHRoZSBlbmQuIiw1NDQsODMyLDEy"
    "NF19LHsiaWQiOjgsInR5cGUiOiJSYW5kb21Ob2lzZSIsInBvcyI6WzAsMTA0MF0sInNpemUiOls0MDAsNjBdLCJmbGFncyI6e30s"
    "Im9yZGVyIjo0LCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiTk9JU0UiLCJ0eXBlIjoiTk9JU0UiLCJs"
    "aW5rcyI6WzddfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJSYW5kb21Ob2lzZSJ9LCJ3aWRnZXRzX3ZhbHVl"
    "cyI6WzQyXX0seyJpZCI6OSwidHlwZSI6IkJhc2ljR3VpZGVyIiwicG9zIjpbMTM4MCwwXSwic2l6ZSI6WzQwMCw4Ml0sImZsYWdz"
    "Ijp7fSwib3JkZXIiOjEwLCJtb2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJtb2RlbCIsInR5cGUiOiJNT0RFTCIsImxpbmsiOjV9"
    "LHsibmFtZSI6ImNvbmRpdGlvbmluZyIsInR5cGUiOiJDT05ESVRJT05JTkciLCJsaW5rIjo2fV0sIm91dHB1dHMiOlt7Im5hbWUi"
    "OiJHVUlERVIiLCJ0eXBlIjoiR1VJREVSIiwibGlua3MiOls4XX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoi"
    "QmFzaWNHdWlkZXIifSwid2lkZ2V0c192YWx1ZXMiOltdfSx7ImlkIjoxMCwidHlwZSI6IktTYW1wbGVyU2VsZWN0IiwicG9zIjpb"
    "MCwxMzAwXSwic2l6ZSI6WzQwMCw2MF0sImZsYWdzIjp7fSwib3JkZXIiOjUsIm1vZGUiOjAsImlucHV0cyI6W10sIm91dHB1dHMi"
    "Olt7Im5hbWUiOiJTQU1QTEVSIiwidHlwZSI6IlNBTVBMRVIiLCJsaW5rcyI6WzldfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1l"
    "IGZvciBTJlIiOiJLU2FtcGxlclNlbGVjdCJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJldWxlciJdfSx7ImlkIjoxMSwidHlwZSI6Ik1h"
    "bnVhbFNpZ21hcyIsInBvcyI6WzAsMTU2MF0sInNpemUiOls0MDAsNjBdLCJmbGFncyI6e30sIm9yZGVyIjo2LCJtb2RlIjowLCJp"
    "bnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiU0lHTUFTIiwidHlwZSI6IlNJR01BUyIsImxpbmtzIjpbMTBdfV0sInByb3Bl"
    "cnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJNYW51YWxTaWdtYXMifSwid2lkZ2V0c192YWx1ZXMiOlsiMC45OTk4OTk5MDk5"
    "LCAwLjk4NTc4ODQwNTEsIDAuOTY3NTc1MjQ4NywgMC45NDMxNjgwNzc0LCAwLjkwOTA5MDkwOTEsIDAuODU3MTQyODU3MSwgMC43"
    "NjkyMzA3NjkyLCAwLjU4ODIzNTI5NDEsIDAiXX0seyJpZCI6MTIsInR5cGUiOiJTYW1wbGVyQ3VzdG9tQWR2YW5jZWQiLCJwb3Mi"
    "OlsxODQwLDBdLCJzaXplIjpbNDAwLDE2MF0sImZsYWdzIjp7fSwib3JkZXIiOjExLCJtb2RlIjowLCJpbnB1dHMiOlt7Im5hbWUi"
    "OiJub2lzZSIsInR5cGUiOiJOT0lTRSIsImxpbmsiOjd9LHsibmFtZSI6Imd1aWRlciIsInR5cGUiOiJHVUlERVIiLCJsaW5rIjo4"
    "fSx7Im5hbWUiOiJzYW1wbGVyIiwidHlwZSI6IlNBTVBMRVIiLCJsaW5rIjo5fSx7Im5hbWUiOiJzaWdtYXMiLCJ0eXBlIjoiU0lH"
    "TUFTIiwibGluayI6MTB9LHsibmFtZSI6ImxhdGVudF9pbWFnZSIsInR5cGUiOiJMQVRFTlQiLCJsaW5rIjoxMX1dLCJvdXRwdXRz"
    "IjpbeyJuYW1lIjoib3V0cHV0IiwidHlwZSI6IkxBVEVOVCIsImxpbmtzIjpbMTIsMTRdfSx7Im5hbWUiOiJkZW5vaXNlZF9vdXRw"
    "dXQiLCJ0eXBlIjoiTEFURU5UIiwibGlua3MiOm51bGx9XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlNhbXBs"
    "ZXJDdXN0b21BZHZhbmNlZCJ9LCJ3aWRnZXRzX3ZhbHVlcyI6W119LHsiaWQiOjEzLCJ0eXBlIjoiVkFFRGVjb2RlIiwicG9zIjpb"
    "MjMwMCwwXSwic2l6ZSI6WzQwMCw4Ml0sImZsYWdzIjp7fSwib3JkZXIiOjEyLCJtb2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJz"
    "YW1wbGVzIiwidHlwZSI6IkxBVEVOVCIsImxpbmsiOjEyfSx7Im5hbWUiOiJ2YWUiLCJ0eXBlIjoiVkFFIiwibGluayI6MTN9XSwi"
    "b3V0cHV0cyI6W3sibmFtZSI6IklNQUdFIiwidHlwZSI6IklNQUdFIiwibGlua3MiOlsxNl19XSwicHJvcGVydGllcyI6eyJOb2Rl"
    "IG5hbWUgZm9yIFMmUiI6IlZBRURlY29kZSJ9LCJ3aWRnZXRzX3ZhbHVlcyI6W119LHsiaWQiOjE0LCJ0eXBlIjoiVkFFRGVjb2Rl"
    "QXVkaW8iLCJwb3MiOlsyMzAwLDI2MF0sInNpemUiOls0MDAsODJdLCJmbGFncyI6e30sIm9yZGVyIjoxMywibW9kZSI6MCwiaW5w"
    "dXRzIjpbeyJuYW1lIjoic2FtcGxlcyIsInR5cGUiOiJMQVRFTlQiLCJsaW5rIjoxNH0seyJuYW1lIjoidmFlIiwidHlwZSI6IlZB"
    "RSIsImxpbmsiOjE1fV0sIm91dHB1dHMiOlt7Im5hbWUiOiJBVURJTyIsInR5cGUiOiJBVURJTyIsImxpbmtzIjpbMTddfV0sInBy"
    "b3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJWQUVEZWNvZGVBdWRpbyJ9LCJ3aWRnZXRzX3ZhbHVlcyI6W119LHsiaWQi"
    "OjE1LCJ0eXBlIjoiQ3JlYXRlVmlkZW8iLCJwb3MiOlsyNzYwLDBdLCJzaXplIjpbNDAwLDE2MF0sImZsYWdzIjp7fSwib3JkZXIi"
    "OjE0LCJtb2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJpbWFnZXMiLCJ0eXBlIjoiSU1BR0UiLCJsaW5rIjoxNn0seyJuYW1lIjoi"
    "YXVkaW8iLCJ0eXBlIjoiQVVESU8iLCJsaW5rIjoxNywic2hhcGUiOjd9XSwib3V0cHV0cyI6W3sibmFtZSI6IlZJREVPIiwidHlw"
    "ZSI6IlZJREVPIiwibGlua3MiOlsxOF19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IkNyZWF0ZVZpZGVvIn0s"
    "IndpZGdldHNfdmFsdWVzIjpbMjQuMCw4LCJzUkdCIl19LHsiaWQiOjE2LCJ0eXBlIjoiU2F2ZVZpZGVvIiwicG9zIjpbMzIyMCww"
    "XSwic2l6ZSI6WzQwMCwxMzRdLCJmbGFncyI6e30sIm9yZGVyIjoxNSwibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoidmlkZW8i"
    "LCJ0eXBlIjoiVklERU8iLCJsaW5rIjoxOH1dLCJvdXRwdXRzIjpbeyJuYW1lIjoidmlkZW8iLCJ0eXBlIjoiVklERU8iLCJsaW5r"
    "cyI6bnVsbH1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiU2F2ZVZpZGVvIn0sIndpZGdldHNfdmFsdWVzIjpb"
    "IkZhc3RIM19TYXlha2FfVDJWQS9GYXN0SDNfVjJfbmV3VkFFXzhzdGVwIiwiYXV0byIsImF1dG8iXX1dLCJsaW5rcyI6W1sxLDEs"
    "MCwyLDAsIk1PREVMIl0sWzIsMiwwLDMsMCwiTU9ERUwiXSxbMyw0LDAsNywwLCJDTElQIl0sWzQsNSwwLDcsMSwiVkFFIl0sWzUs"
    "MywwLDksMCwiTU9ERUwiXSxbNiw3LDAsOSwxLCJDT05ESVRJT05JTkciXSxbNyw4LDAsMTIsMCwiTk9JU0UiXSxbOCw5LDAsMTIs"
    "MSwiR1VJREVSIl0sWzksMTAsMCwxMiwyLCJTQU1QTEVSIl0sWzEwLDExLDAsMTIsMywiU0lHTUFTIl0sWzExLDcsMSwxMiw0LCJM"
    "QVRFTlQiXSxbMTIsMTIsMCwxMywwLCJMQVRFTlQiXSxbMTMsNSwwLDEzLDEsIlZBRSJdLFsxNCwxMiwwLDE0LDAsIkxBVEVOVCJd"
    "LFsxNSw2LDAsMTQsMSwiVkFFIl0sWzE2LDEzLDAsMTUsMCwiSU1BR0UiXSxbMTcsMTQsMCwxNSwxLCJBVURJTyJdLFsxOCwxNSww"
    "LDE2LDAsIlZJREVPIl1dLCJncm91cHMiOltdLCJjb25maWciOnt9LCJleHRyYSI6eyJkcyI6eyJzY2FsZSI6MC43LCJvZmZzZXQi"
    "OlswLDBdfX0sInZlcnNpb24iOjAuNH0="
)
_WF_I2V_B64 = (
    "eyJpZCI6ImZhc3RoMy1pMnYiLCJyZXZpc2lvbiI6MCwibGFzdF9ub2RlX2lkIjoxNywibGFzdF9saW5rX2lkIjoxOSwibm9kZXMi"
    "Olt7ImlkIjoxLCJ0eXBlIjoiVU5FVExvYWRlciIsInBvcyI6WzAsMF0sInNpemUiOls0MDAsODJdLCJmbGFncyI6e30sIm9yZGVy"
    "IjowLCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiTU9ERUwiLCJ0eXBlIjoiTU9ERUwiLCJsaW5rcyI6"
    "WzFdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJVTkVUTG9hZGVyIn0sIndpZGdldHNfdmFsdWVzIjpbImZh"
    "c3R2aWRlb19mYXN0aDNfOHN0ZXBfdjJfcHJ1bmVkX2ludDhfY29udnJvdC5zYWZldGVuc29ycyIsImRlZmF1bHQiXX0seyJpZCI6"
    "MiwidHlwZSI6Ik1pbmlNYXhIM1NpZ21hU2hpZnQiLCJwb3MiOls0NjAsMF0sInNpemUiOls0MDAsMTA4XSwiZmxhZ3MiOnt9LCJv"
    "cmRlciI6OCwibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoibW9kZWwiLCJ0eXBlIjoiTU9ERUwiLCJsaW5rIjoxfV0sIm91dHB1"
    "dHMiOlt7Im5hbWUiOiJNT0RFTCIsInR5cGUiOiJNT0RFTCIsImxpbmtzIjpbMl19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUg"
    "Zm9yIFMmUiI6Ik1pbmlNYXhIM1NpZ21hU2hpZnQifSwid2lkZ2V0c192YWx1ZXMiOlsxMC4wLDMuMF19LHsiaWQiOjMsInR5cGUi"
    "OiJCbG9ja1NwYXJzZUF0dGVudGlvbiIsInBvcyI6WzkyMCwwXSwic2l6ZSI6WzQwMCwzNTBdLCJmbGFncyI6e30sIm9yZGVyIjox"
    "MCwibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoibW9kZWwiLCJ0eXBlIjoiTU9ERUwiLCJsaW5rIjoyfV0sIm91dHB1dHMiOlt7"
    "Im5hbWUiOiJNT0RFTCIsInR5cGUiOiJNT0RFTCIsImxpbmtzIjpbNl19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMm"
    "UiI6IkJsb2NrU3BhcnNlQXR0ZW50aW9uIn0sIndpZGdldHNfdmFsdWVzIjpbInZzYSIsMjAsMCwxLCIiLDAsMCwiZXhhY3Rfa3Zf"
    "YW5kX3Jvd3MiLHRydWVdLCJ3aWRnZXRzX3ZhbHVlc19uYW1lZCI6eyJzZWxlY3Rpb24ua2VlcF9wZXJjZW50IjoyMCwiZW5kX3Bl"
    "cmNlbnQiOjEsImV4dHJhX3Rva2VucyI6MCwic3RhcnRfcGVyY2VudCI6MCwiZGVuc2VfYmxvY2tzIjoiIiwic2VsZWN0aW9uIjoi"
    "dnNhIiwidmVyYm9zZSI6dHJ1ZSwic2lua19jb25kaXRpb25pbmciOiJleGFjdF9rdl9hbmRfcm93cyIsIm1pbl90b2tlbnMiOjB9"
    "fSx7ImlkIjo0LCJ0eXBlIjoiQ0xJUExvYWRlciIsInBvcyI6WzAsMjYwXSwic2l6ZSI6WzQwMCwxMDhdLCJmbGFncyI6e30sIm9y"
    "ZGVyIjoxLCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiQ0xJUCIsInR5cGUiOiJDTElQIiwibGlua3Mi"
    "OlszXX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiQ0xJUExvYWRlciJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJx"
    "d2VuM3ZsXzMyYl9taW5pbWF4X2gzX252ZnA0X2F3cS5zYWZldGVuc29ycyIsIm1pbmltYXgiLCJkZWZhdWx0Il19LHsiaWQiOjUs"
    "InR5cGUiOiJWQUVMb2FkZXIiLCJwb3MiOlswLDUyMF0sInNpemUiOls0MDAsNjBdLCJmbGFncyI6e30sIm9yZGVyIjoyLCJtb2Rl"
    "IjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiVkFFIiwidHlwZSI6IlZBRSIsImxpbmtzIjpbNCwxNF19XSwicHJv"
    "cGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlZBRUxvYWRlciJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJtaW5pbWF4X2gzX3Zp"
    "ZGVvX3ZhZV9pbnQ4X2NvbnZyb3Quc2FmZXRlbnNvcnMiXX0seyJpZCI6NiwidHlwZSI6IlZBRUxvYWRlciIsInBvcyI6WzAsNzgw"
    "XSwic2l6ZSI6WzQwMCw2MF0sImZsYWdzIjp7fSwib3JkZXIiOjMsIm1vZGUiOjAsImlucHV0cyI6W10sIm91dHB1dHMiOlt7Im5h"
    "bWUiOiJWQUUiLCJ0eXBlIjoiVkFFIiwibGlua3MiOlsxNl19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlZB"
    "RUxvYWRlciJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJtaW5pbWF4X2gzX2F1ZGlvX3ZhZV9mcDMyLnNhZmV0ZW5zb3JzIl19LHsiaWQi"
    "OjcsInR5cGUiOiJNaW5pTWF4SDNJbWFnZVRvVmlkZW8iLCJwb3MiOls0NjAsMjYwXSwic2l6ZSI6WzQwMCwyMzhdLCJmbGFncyI6"
    "e30sIm9yZGVyIjo5LCJtb2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJjbGlwIiwidHlwZSI6IkNMSVAiLCJsaW5rIjozfSx7Im5h"
    "bWUiOiJ2YWUiLCJ0eXBlIjoiVkFFIiwibGluayI6NH0seyJuYW1lIjoiZmlyc3RfZnJhbWUiLCJ0eXBlIjoiSU1BR0UiLCJsaW5r"
    "Ijo1LCJzaGFwZSI6N30seyJuYW1lIjoibGFzdF9mcmFtZSIsInR5cGUiOiJJTUFHRSIsImxpbmsiOm51bGwsInNoYXBlIjo3fV0s"
    "Im91dHB1dHMiOlt7Im5hbWUiOiJwb3NpdGl2ZSIsInR5cGUiOiJDT05ESVRJT05JTkciLCJsaW5rcyI6WzddfSx7Im5hbWUiOiJM"
    "QVRFTlQiLCJ0eXBlIjoiTEFURU5UIiwibGlua3MiOlsxMl19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6Ik1p"
    "bmlNYXhIM0ltYWdlVG9WaWRlbyJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJpbnRlZ3JhdGVkX211bHRpbW9kYWxfZGVzY3JpcHRpb246"
    "IFtTaG90IDFdIDJELWFuaW1hdGVkLCBhbmltZSBzdHlsZSwgYSBtZWRpdW0gY2xvc2UtdXAgc2hvdCBmcmFtZXMgYSBjaGVlcmZ1"
    "bCB5b3VuZyB3b21hbiB3aXRoIGRhcmsgaGFpciB0aWVkIHVwIGluIGEgbmVhdCBidW4sIHNpZGUgYmFuZ3MgZnJhbWluZyBoZXIg"
    "ZGVsaWNhdGUgZmFjZSwgYW5kIGRhcmsgcHVycGxlIGV5ZXMgd2VhcmluZyBzbWFsbCBwZWFybCBlYXJyaW5ncy4gU2hlIHdlYXJz"
    "IGEgbGlnaHQgZ3JheSB0dXJ0bGVuZWNrIGtuaXQgdG9wIHVuZGVyIGEgYmxhY2sgc3BhZ2hldHRpLXN0cmFwIG1pbmkgZHJlc3Mu"
    "IFN0YW5kaW5nIGluIGEgY2xlYW4gbW9kZXJuIHJvb20gd2l0aCB3aGl0ZSB3YWxscyBhbmQgYSBkYXJrIGRvb3JmcmFtZSwgc2hl"
    "IHRpbHRzIGhlciBoZWFkIHNsaWdodGx5IHdpdGggYSBicmlnaHQgc21pbGUsIGxpZ2h0bHkgcmVzdGluZyBoZXIgcmlnaHQgZmlu"
    "Z2VydGlwcyBuZWFyIGhlciBjb2xsYXJib25lLiBUaGUgY2FtZXJhIHB1c2hlcyBpbiB3aXRoIHNtYWxsIGFtcGxpdHVkZSBhdCBz"
    "bG93IHNwZWVkIGFzIHRoZSB5b3VuZyB3b21hbiB3aXRoIGEgc3dlZXQsIGNsZWFyIHZvaWNlIChTMSkgZ2VudGx5IHNheXM6IDxk"
    "PltKYXBhbmVzZV0g44GT44KT44Gr44Gh44Gv77yB5LuK5pel44KC5LiA5pel6aCR5by144KN44GG44Gt44CCPC9kPlxuXG5vdmVy"
    "YWxsX3NvdW5kc2NhcGU6IFF1aWV0IGluZG9vciByb29tIHRvbmUsIGEgc29mdCBydXN0bGUgb2Yga25pdCBmYWJyaWMgYXMgc2hl"
    "IG1vdmVzIGhlciBzaG91bGRlciwgYW5kIHRoZSBjbGVhciBzcG9rZW4gdm9pY2UgaW4gYSBuYXR1cmFsIHJvb20gYWNvdXN0aWMu"
    "XG5cbm5vbl9kaWVnZXRpY19tdXNpYzogQSBnZW50bGUsIHVwbGlmdGluZyBhY291c3RpYyBndWl0YXIgcGF0dGVybiB3aXRoIHdh"
    "cm0gcGlhbm8gbm90ZXMgYXQgYSBtb2RlcmF0ZSB0ZW1wbywgZmFkaW5nIHNvZnRseSBhdCB0aGUgZW5kLiIsNTQ0LDgzMiwxMjRd"
    "fSx7ImlkIjo4LCJ0eXBlIjoiUmFuZG9tTm9pc2UiLCJwb3MiOlswLDEwNDBdLCJzaXplIjpbNDAwLDYwXSwiZmxhZ3MiOnt9LCJv"
    "cmRlciI6NCwibW9kZSI6MCwiaW5wdXRzIjpbXSwib3V0cHV0cyI6W3sibmFtZSI6Ik5PSVNFIiwidHlwZSI6Ik5PSVNFIiwibGlu"
    "a3MiOls4XX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiUmFuZG9tTm9pc2UifSwid2lkZ2V0c192YWx1ZXMi"
    "Ols0Ml19LHsiaWQiOjksInR5cGUiOiJCYXNpY0d1aWRlciIsInBvcyI6WzEzODAsMF0sInNpemUiOls0MDAsODJdLCJmbGFncyI6"
    "e30sIm9yZGVyIjoxMSwibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoibW9kZWwiLCJ0eXBlIjoiTU9ERUwiLCJsaW5rIjo2fSx7"
    "Im5hbWUiOiJjb25kaXRpb25pbmciLCJ0eXBlIjoiQ09ORElUSU9OSU5HIiwibGluayI6N31dLCJvdXRwdXRzIjpbeyJuYW1lIjoi"
    "R1VJREVSIiwidHlwZSI6IkdVSURFUiIsImxpbmtzIjpbOV19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IkJh"
    "c2ljR3VpZGVyIn0sIndpZGdldHNfdmFsdWVzIjpbXX0seyJpZCI6MTAsInR5cGUiOiJLU2FtcGxlclNlbGVjdCIsInBvcyI6WzAs"
    "MTMwMF0sInNpemUiOls0MDAsNjBdLCJmbGFncyI6e30sIm9yZGVyIjo1LCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpb"
    "eyJuYW1lIjoiU0FNUExFUiIsInR5cGUiOiJTQU1QTEVSIiwibGlua3MiOlsxMF19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUg"
    "Zm9yIFMmUiI6IktTYW1wbGVyU2VsZWN0In0sIndpZGdldHNfdmFsdWVzIjpbImV1bGVyIl19LHsiaWQiOjExLCJ0eXBlIjoiTWFu"
    "dWFsU2lnbWFzIiwicG9zIjpbMCwxNTYwXSwic2l6ZSI6WzQwMCw2MF0sImZsYWdzIjp7fSwib3JkZXIiOjYsIm1vZGUiOjAsImlu"
    "cHV0cyI6W10sIm91dHB1dHMiOlt7Im5hbWUiOiJTSUdNQVMiLCJ0eXBlIjoiU0lHTUFTIiwibGlua3MiOlsxMV19XSwicHJvcGVy"
    "dGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6Ik1hbnVhbFNpZ21hcyJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyIwLjk5OTg5OTkwOTks"
    "IDAuOTg1Nzg4NDA1MSwgMC45Njc1NzUyNDg3LCAwLjk0MzE2ODA3NzQsIDAuOTA5MDkwOTA5MSwgMC44NTcxNDI4NTcxLCAwLjc2"
    "OTIzMDc2OTIsIDAuNTg4MjM1Mjk0MSwgMCJdfSx7ImlkIjoxMiwidHlwZSI6IlNhbXBsZXJDdXN0b21BZHZhbmNlZCIsInBvcyI6"
    "WzE4NDAsMF0sInNpemUiOls0MDAsMTYwXSwiZmxhZ3MiOnt9LCJvcmRlciI6MTIsIm1vZGUiOjAsImlucHV0cyI6W3sibmFtZSI6"
    "Im5vaXNlIiwidHlwZSI6Ik5PSVNFIiwibGluayI6OH0seyJuYW1lIjoiZ3VpZGVyIiwidHlwZSI6IkdVSURFUiIsImxpbmsiOjl9"
    "LHsibmFtZSI6InNhbXBsZXIiLCJ0eXBlIjoiU0FNUExFUiIsImxpbmsiOjEwfSx7Im5hbWUiOiJzaWdtYXMiLCJ0eXBlIjoiU0lH"
    "TUFTIiwibGluayI6MTF9LHsibmFtZSI6ImxhdGVudF9pbWFnZSIsInR5cGUiOiJMQVRFTlQiLCJsaW5rIjoxMn1dLCJvdXRwdXRz"
    "IjpbeyJuYW1lIjoib3V0cHV0IiwidHlwZSI6IkxBVEVOVCIsImxpbmtzIjpbMTMsMTVdfSx7Im5hbWUiOiJkZW5vaXNlZF9vdXRw"
    "dXQiLCJ0eXBlIjoiTEFURU5UIiwibGlua3MiOm51bGx9XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlNhbXBs"
    "ZXJDdXN0b21BZHZhbmNlZCJ9LCJ3aWRnZXRzX3ZhbHVlcyI6W119LHsiaWQiOjEzLCJ0eXBlIjoiVkFFRGVjb2RlIiwicG9zIjpb"
    "MjMwMCwwXSwic2l6ZSI6WzQwMCw4Ml0sImZsYWdzIjp7fSwib3JkZXIiOjEzLCJtb2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJz"
    "YW1wbGVzIiwidHlwZSI6IkxBVEVOVCIsImxpbmsiOjEzfSx7Im5hbWUiOiJ2YWUiLCJ0eXBlIjoiVkFFIiwibGluayI6MTR9XSwi"
    "b3V0cHV0cyI6W3sibmFtZSI6IklNQUdFIiwidHlwZSI6IklNQUdFIiwibGlua3MiOlsxN119XSwicHJvcGVydGllcyI6eyJOb2Rl"
    "IG5hbWUgZm9yIFMmUiI6IlZBRURlY29kZSJ9LCJ3aWRnZXRzX3ZhbHVlcyI6W119LHsiaWQiOjE0LCJ0eXBlIjoiVkFFRGVjb2Rl"
    "QXVkaW8iLCJwb3MiOlsyMzAwLDI2MF0sInNpemUiOls0MDAsODJdLCJmbGFncyI6e30sIm9yZGVyIjoxNCwibW9kZSI6MCwiaW5w"
    "dXRzIjpbeyJuYW1lIjoic2FtcGxlcyIsInR5cGUiOiJMQVRFTlQiLCJsaW5rIjoxNX0seyJuYW1lIjoidmFlIiwidHlwZSI6IlZB"
    "RSIsImxpbmsiOjE2fV0sIm91dHB1dHMiOlt7Im5hbWUiOiJBVURJTyIsInR5cGUiOiJBVURJTyIsImxpbmtzIjpbMThdfV0sInBy"
    "b3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJWQUVEZWNvZGVBdWRpbyJ9LCJ3aWRnZXRzX3ZhbHVlcyI6W119LHsiaWQi"
    "OjE1LCJ0eXBlIjoiQ3JlYXRlVmlkZW8iLCJwb3MiOlsyNzYwLDBdLCJzaXplIjpbNDAwLDE2MF0sImZsYWdzIjp7fSwib3JkZXIi"
    "OjE1LCJtb2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJpbWFnZXMiLCJ0eXBlIjoiSU1BR0UiLCJsaW5rIjoxN30seyJuYW1lIjoi"
    "YXVkaW8iLCJ0eXBlIjoiQVVESU8iLCJsaW5rIjoxOCwic2hhcGUiOjd9XSwib3V0cHV0cyI6W3sibmFtZSI6IlZJREVPIiwidHlw"
    "ZSI6IlZJREVPIiwibGlua3MiOlsxOV19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IkNyZWF0ZVZpZGVvIn0s"
    "IndpZGdldHNfdmFsdWVzIjpbMjQuMCw4LCJzUkdCIl19LHsiaWQiOjE2LCJ0eXBlIjoiU2F2ZVZpZGVvIiwicG9zIjpbMzIyMCww"
    "XSwic2l6ZSI6WzQwMCwxMzRdLCJmbGFncyI6e30sIm9yZGVyIjoxNiwibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoidmlkZW8i"
    "LCJ0eXBlIjoiVklERU8iLCJsaW5rIjoxOX1dLCJvdXRwdXRzIjpbeyJuYW1lIjoidmlkZW8iLCJ0eXBlIjoiVklERU8iLCJsaW5r"
    "cyI6bnVsbH1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiU2F2ZVZpZGVvIn0sIndpZGdldHNfdmFsdWVzIjpb"
    "IkZhc3RIMy9mYXN0aDNfdjJfbmV3dmFlX2kydiIsImF1dG8iLCJhdXRvIl19LHsiaWQiOjE3LCJ0eXBlIjoiTG9hZEltYWdlIiwi"
    "cG9zIjpbMCwxODIwXSwic2l6ZSI6WzQwMCw2MF0sImZsYWdzIjp7fSwib3JkZXIiOjcsIm1vZGUiOjAsImlucHV0cyI6W10sIm91"
    "dHB1dHMiOlt7Im5hbWUiOiJJTUFHRSIsInR5cGUiOiJJTUFHRSIsImxpbmtzIjpbNV19LHsibmFtZSI6Ik1BU0siLCJ0eXBlIjoi"
    "TUFTSyIsImxpbmtzIjpudWxsfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJMb2FkSW1hZ2UifSwid2lkZ2V0"
    "c192YWx1ZXMiOlsicG9ydHJhaXRfcGhvdG9fbWFuX2IucG5nIl19XSwibGlua3MiOltbMSwxLDAsMiwwLCJNT0RFTCJdLFsyLDIs"
    "MCwzLDAsIk1PREVMIl0sWzMsNCwwLDcsMCwiQ0xJUCJdLFs0LDUsMCw3LDEsIlZBRSJdLFs1LDE3LDAsNywyLCJJTUFHRSJdLFs2"
    "LDMsMCw5LDAsIk1PREVMIl0sWzcsNywwLDksMSwiQ09ORElUSU9OSU5HIl0sWzgsOCwwLDEyLDAsIk5PSVNFIl0sWzksOSwwLDEy"
    "LDEsIkdVSURFUiJdLFsxMCwxMCwwLDEyLDIsIlNBTVBMRVIiXSxbMTEsMTEsMCwxMiwzLCJTSUdNQVMiXSxbMTIsNywxLDEyLDQs"
    "IkxBVEVOVCJdLFsxMywxMiwwLDEzLDAsIkxBVEVOVCJdLFsxNCw1LDAsMTMsMSwiVkFFIl0sWzE1LDEyLDAsMTQsMCwiTEFURU5U"
    "Il0sWzE2LDYsMCwxNCwxLCJWQUUiXSxbMTcsMTMsMCwxNSwwLCJJTUFHRSJdLFsxOCwxNCwwLDE1LDEsIkFVRElPIl0sWzE5LDE1"
    "LDAsMTYsMCwiVklERU8iXV0sImdyb3VwcyI6W10sImNvbmZpZyI6e30sImV4dHJhIjp7ImRzIjp7InNjYWxlIjowLjcsIm9mZnNl"
    "dCI6WzAsMF19fSwidmVyc2lvbiI6MC40fQ=="
)
_WF_R2V_B64 = (
    "eyJpZCI6ImZhc3RoMy1yMnYiLCJyZXZpc2lvbiI6MCwibGFzdF9ub2RlX2lkIjoxNywibGFzdF9saW5rX2lkIjoyMCwibm9kZXMi"
    "Olt7ImlkIjoxLCJ0eXBlIjoiVU5FVExvYWRlciIsInBvcyI6WzAsMF0sInNpemUiOls0MDAsODJdLCJmbGFncyI6e30sIm9yZGVy"
    "IjowLCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiTU9ERUwiLCJ0eXBlIjoiTU9ERUwiLCJsaW5rcyI6"
    "WzFdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJVTkVUTG9hZGVyIn0sIndpZGdldHNfdmFsdWVzIjpbImZh"
    "c3R2aWRlb19mYXN0aDNfOHN0ZXBfdjJfcHJ1bmVkX2ludDhfY29udnJvdC5zYWZldGVuc29ycyIsImRlZmF1bHQiXX0seyJpZCI6"
    "MiwidHlwZSI6Ik1pbmlNYXhIM1NpZ21hU2hpZnQiLCJwb3MiOls0NjAsMF0sInNpemUiOls0MDAsMTA4XSwiZmxhZ3MiOnt9LCJv"
    "cmRlciI6OCwibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoibW9kZWwiLCJ0eXBlIjoiTU9ERUwiLCJsaW5rIjoxfV0sIm91dHB1"
    "dHMiOlt7Im5hbWUiOiJNT0RFTCIsInR5cGUiOiJNT0RFTCIsImxpbmtzIjpbMl19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUg"
    "Zm9yIFMmUiI6Ik1pbmlNYXhIM1NpZ21hU2hpZnQifSwid2lkZ2V0c192YWx1ZXMiOlsxMC4wLDMuMF19LHsiaWQiOjMsInR5cGUi"
    "OiJCbG9ja1NwYXJzZUF0dGVudGlvbiIsInBvcyI6WzkyMCwwXSwic2l6ZSI6WzQwMCwzNTBdLCJmbGFncyI6e30sIm9yZGVyIjox"
    "MCwibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoibW9kZWwiLCJ0eXBlIjoiTU9ERUwiLCJsaW5rIjoyfV0sIm91dHB1dHMiOlt7"
    "Im5hbWUiOiJNT0RFTCIsInR5cGUiOiJNT0RFTCIsImxpbmtzIjpbNl19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMm"
    "UiI6IkJsb2NrU3BhcnNlQXR0ZW50aW9uIn0sIndpZGdldHNfdmFsdWVzIjpbInZzYSIsMjAsMCwxLCIiLDAsMCwiZXhhY3Rfa3Zf"
    "YW5kX3Jvd3MiLHRydWVdLCJ3aWRnZXRzX3ZhbHVlc19uYW1lZCI6eyJzZWxlY3Rpb24ua2VlcF9wZXJjZW50IjoyMCwiZW5kX3Bl"
    "cmNlbnQiOjEsImV4dHJhX3Rva2VucyI6MCwic3RhcnRfcGVyY2VudCI6MCwiZGVuc2VfYmxvY2tzIjoiIiwic2VsZWN0aW9uIjoi"
    "dnNhIiwidmVyYm9zZSI6dHJ1ZSwic2lua19jb25kaXRpb25pbmciOiJleGFjdF9rdl9hbmRfcm93cyIsIm1pbl90b2tlbnMiOjB9"
    "fSx7ImlkIjo0LCJ0eXBlIjoiQ0xJUExvYWRlciIsInBvcyI6WzAsMjYwXSwic2l6ZSI6WzQwMCwxMDhdLCJmbGFncyI6e30sIm9y"
    "ZGVyIjoxLCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiQ0xJUCIsInR5cGUiOiJDTElQIiwibGlua3Mi"
    "OlszXX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiQ0xJUExvYWRlciJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJx"
    "d2VuM3ZsXzMyYl9taW5pbWF4X2gzX252ZnA0X2F3cS5zYWZldGVuc29ycyIsIm1pbmltYXgiLCJkZWZhdWx0Il19LHsiaWQiOjUs"
    "InR5cGUiOiJWQUVMb2FkZXIiLCJwb3MiOlswLDUyMF0sInNpemUiOls0MDAsNjBdLCJmbGFncyI6e30sIm9yZGVyIjoyLCJtb2Rl"
    "IjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiVkFFIiwidHlwZSI6IlZBRSIsImxpbmtzIjpbNCwxNF19XSwicHJv"
    "cGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlZBRUxvYWRlciJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJtaW5pbWF4X2gzX3Zp"
    "ZGVvX3ZhZV9pbnQ4X2NvbnZyb3Quc2FmZXRlbnNvcnMiXX0seyJpZCI6NiwidHlwZSI6IlZBRUxvYWRlciIsInBvcyI6WzAsNzgw"
    "XSwic2l6ZSI6WzQwMCw2MF0sImZsYWdzIjp7fSwib3JkZXIiOjMsIm1vZGUiOjAsImlucHV0cyI6W10sIm91dHB1dHMiOlt7Im5h"
    "bWUiOiJWQUUiLCJ0eXBlIjoiVkFFIiwibGlua3MiOls1LDE2XX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoi"
    "VkFFTG9hZGVyIn0sIndpZGdldHNfdmFsdWVzIjpbIm1pbmltYXhfaDNfYXVkaW9fdmFlX2ZwMzIuc2FmZXRlbnNvcnMiXX0seyJp"
    "ZCI6NywidHlwZSI6Ik1pbmlNYXhIM1JlZmVyZW5jZVRvVmlkZW8iLCJwb3MiOls0NjAsMjYwXSwic2l6ZSI6WzQwMCwzMTZdLCJm"
    "bGFncyI6e30sIm9yZGVyIjo5LCJtb2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJjbGlwIiwidHlwZSI6IkNMSVAiLCJsaW5rIjoz"
    "fSx7Im5hbWUiOiJ2YWUiLCJ0eXBlIjoiVkFFIiwibGluayI6NH0seyJuYW1lIjoiYXVkaW9fdmFlIiwidHlwZSI6IlZBRSIsImxp"
    "bmsiOjV9LHsibmFtZSI6InJlZl9pbWFnZXMucmVmX2ltYWdlXzAiLCJ0eXBlIjoiSU1BR0UiLCJsaW5rIjoyMH1dLCJvdXRwdXRz"
    "IjpbeyJuYW1lIjoicG9zaXRpdmUiLCJ0eXBlIjoiQ09ORElUSU9OSU5HIiwibGlua3MiOls3XX0seyJuYW1lIjoiTEFURU5UIiwi"
    "dHlwZSI6IkxBVEVOVCIsImxpbmtzIjpbMTJdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJNaW5pTWF4SDNS"
    "ZWZlcmVuY2VUb1ZpZGVvIn0sIndpZGdldHNfdmFsdWVzIjpbImludGVncmF0ZWRfbXVsdGltb2RhbF9kZXNjcmlwdGlvbjogW1No"
    "b3QgMV0gMkQtYW5pbWF0ZWQsIGFuaW1lIHN0eWxlLCBhIG1lZGl1bSBjbG9zZS11cCBzaG90IGZyYW1lcyBhIGNoZWVyZnVsIHlv"
    "dW5nIHdvbWFuIHdpdGggZGFyayBoYWlyIHRpZWQgdXAgaW4gYSBuZWF0IGJ1biwgc2lkZSBiYW5ncyBmcmFtaW5nIGhlciBkZWxp"
    "Y2F0ZSBmYWNlLCBhbmQgZGFyayBwdXJwbGUgZXllcyB3ZWFyaW5nIHNtYWxsIHBlYXJsIGVhcnJpbmdzLiBTaGUgd2VhcnMgYSBs"
    "aWdodCBncmF5IHR1cnRsZW5lY2sga25pdCB0b3AgdW5kZXIgYSBibGFjayBzcGFnaGV0dGktc3RyYXAgbWluaSBkcmVzcy4gU3Rh"
    "bmRpbmcgaW4gYSBjbGVhbiBtb2Rlcm4gcm9vbSB3aXRoIHdoaXRlIHdhbGxzIGFuZCBhIGRhcmsgZG9vcmZyYW1lLCBzaGUgdGls"
    "dHMgaGVyIGhlYWQgc2xpZ2h0bHkgd2l0aCBhIGJyaWdodCBzbWlsZSwgbGlnaHRseSByZXN0aW5nIGhlciByaWdodCBmaW5nZXJ0"
    "aXBzIG5lYXIgaGVyIGNvbGxhcmJvbmUuIFRoZSBjYW1lcmEgcHVzaGVzIGluIHdpdGggc21hbGwgYW1wbGl0dWRlIGF0IHNsb3cg"
    "c3BlZWQgYXMgdGhlIHlvdW5nIHdvbWFuIHdpdGggYSBzd2VldCwgY2xlYXIgdm9pY2UgKFMxKSBnZW50bHkgc2F5czogPGQ+W0ph"
    "cGFuZXNlXSDjgZPjgpPjgavjgaHjga/vvIHku4rml6XjgoLkuIDml6XpoJHlvLXjgo3jgYbjga3jgII8L2Q+XG5cbm92ZXJhbGxf"
    "c291bmRzY2FwZTogUXVpZXQgaW5kb29yIHJvb20gdG9uZSwgYSBzb2Z0IHJ1c3RsZSBvZiBrbml0IGZhYnJpYyBhcyBzaGUgbW92"
    "ZXMgaGVyIHNob3VsZGVyLCBhbmQgdGhlIGNsZWFyIHNwb2tlbiB2b2ljZSBpbiBhIG5hdHVyYWwgcm9vbSBhY291c3RpYy5cblxu"
    "bm9uX2RpZWdldGljX211c2ljOiBBIGdlbnRsZSwgdXBsaWZ0aW5nIGFjb3VzdGljIGd1aXRhciBwYXR0ZXJuIHdpdGggd2FybSBw"
    "aWFubyBub3RlcyBhdCBhIG1vZGVyYXRlIHRlbXBvLCBmYWRpbmcgc29mdGx5IGF0IHRoZSBlbmQuIiw1NDQsODMyLDEyNCwibWF4"
    "IixudWxsLG51bGwsbnVsbF19LHsiaWQiOjgsInR5cGUiOiJSYW5kb21Ob2lzZSIsInBvcyI6WzAsMTA0MF0sInNpemUiOls0MDAs"
    "NjBdLCJmbGFncyI6e30sIm9yZGVyIjo0LCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiTk9JU0UiLCJ0"
    "eXBlIjoiTk9JU0UiLCJsaW5rcyI6WzhdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJSYW5kb21Ob2lzZSJ9"
    "LCJ3aWRnZXRzX3ZhbHVlcyI6WzQyXX0seyJpZCI6OSwidHlwZSI6IkJhc2ljR3VpZGVyIiwicG9zIjpbMTM4MCwwXSwic2l6ZSI6"
    "WzQwMCw4Ml0sImZsYWdzIjp7fSwib3JkZXIiOjExLCJtb2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJtb2RlbCIsInR5cGUiOiJN"
    "T0RFTCIsImxpbmsiOjZ9LHsibmFtZSI6ImNvbmRpdGlvbmluZyIsInR5cGUiOiJDT05ESVRJT05JTkciLCJsaW5rIjo3fV0sIm91"
    "dHB1dHMiOlt7Im5hbWUiOiJHVUlERVIiLCJ0eXBlIjoiR1VJREVSIiwibGlua3MiOls5XX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUg"
    "bmFtZSBmb3IgUyZSIjoiQmFzaWNHdWlkZXIifSwid2lkZ2V0c192YWx1ZXMiOltdfSx7ImlkIjoxMCwidHlwZSI6IktTYW1wbGVy"
    "U2VsZWN0IiwicG9zIjpbMCwxMzAwXSwic2l6ZSI6WzQwMCw2MF0sImZsYWdzIjp7fSwib3JkZXIiOjUsIm1vZGUiOjAsImlucHV0"
    "cyI6W10sIm91dHB1dHMiOlt7Im5hbWUiOiJTQU1QTEVSIiwidHlwZSI6IlNBTVBMRVIiLCJsaW5rcyI6WzEwXX1dLCJwcm9wZXJ0"
    "aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiS1NhbXBsZXJTZWxlY3QifSwid2lkZ2V0c192YWx1ZXMiOlsiZXVsZXIiXX0seyJp"
    "ZCI6MTEsInR5cGUiOiJNYW51YWxTaWdtYXMiLCJwb3MiOlswLDE1NjBdLCJzaXplIjpbNDAwLDYwXSwiZmxhZ3MiOnt9LCJvcmRl"
    "ciI6NiwibW9kZSI6MCwiaW5wdXRzIjpbXSwib3V0cHV0cyI6W3sibmFtZSI6IlNJR01BUyIsInR5cGUiOiJTSUdNQVMiLCJsaW5r"
    "cyI6WzExXX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiTWFudWFsU2lnbWFzIn0sIndpZGdldHNfdmFsdWVz"
    "IjpbIjAuOTk5ODk5OTA5OSwgMC45ODU3ODg0MDUxLCAwLjk2NzU3NTI0ODcsIDAuOTQzMTY4MDc3NCwgMC45MDkwOTA5MDkxLCAw"
    "Ljg1NzE0Mjg1NzEsIDAuNzY5MjMwNzY5MiwgMC41ODgyMzUyOTQxLCAwIl19LHsiaWQiOjEyLCJ0eXBlIjoiU2FtcGxlckN1c3Rv"
    "bUFkdmFuY2VkIiwicG9zIjpbMTg0MCwwXSwic2l6ZSI6WzQwMCwxNjBdLCJmbGFncyI6e30sIm9yZGVyIjoxMiwibW9kZSI6MCwi"
    "aW5wdXRzIjpbeyJuYW1lIjoibm9pc2UiLCJ0eXBlIjoiTk9JU0UiLCJsaW5rIjo4fSx7Im5hbWUiOiJndWlkZXIiLCJ0eXBlIjoi"
    "R1VJREVSIiwibGluayI6OX0seyJuYW1lIjoic2FtcGxlciIsInR5cGUiOiJTQU1QTEVSIiwibGluayI6MTB9LHsibmFtZSI6InNp"
    "Z21hcyIsInR5cGUiOiJTSUdNQVMiLCJsaW5rIjoxMX0seyJuYW1lIjoibGF0ZW50X2ltYWdlIiwidHlwZSI6IkxBVEVOVCIsImxp"
    "bmsiOjEyfV0sIm91dHB1dHMiOlt7Im5hbWUiOiJvdXRwdXQiLCJ0eXBlIjoiTEFURU5UIiwibGlua3MiOlsxMywxNV19LHsibmFt"
    "ZSI6ImRlbm9pc2VkX291dHB1dCIsInR5cGUiOiJMQVRFTlQiLCJsaW5rcyI6bnVsbH1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFt"
    "ZSBmb3IgUyZSIjoiU2FtcGxlckN1c3RvbUFkdmFuY2VkIn0sIndpZGdldHNfdmFsdWVzIjpbXX0seyJpZCI6MTMsInR5cGUiOiJW"
    "QUVEZWNvZGUiLCJwb3MiOlsyMzAwLDBdLCJzaXplIjpbNDAwLDgyXSwiZmxhZ3MiOnt9LCJvcmRlciI6MTMsIm1vZGUiOjAsImlu"
    "cHV0cyI6W3sibmFtZSI6InNhbXBsZXMiLCJ0eXBlIjoiTEFURU5UIiwibGluayI6MTN9LHsibmFtZSI6InZhZSIsInR5cGUiOiJW"
    "QUUiLCJsaW5rIjoxNH1dLCJvdXRwdXRzIjpbeyJuYW1lIjoiSU1BR0UiLCJ0eXBlIjoiSU1BR0UiLCJsaW5rcyI6WzE3XX1dLCJw"
    "cm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiVkFFRGVjb2RlIn0sIndpZGdldHNfdmFsdWVzIjpbXX0seyJpZCI6MTQs"
    "InR5cGUiOiJWQUVEZWNvZGVBdWRpbyIsInBvcyI6WzIzMDAsMjYwXSwic2l6ZSI6WzQwMCw4Ml0sImZsYWdzIjp7fSwib3JkZXIi"
    "OjE0LCJtb2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJzYW1wbGVzIiwidHlwZSI6IkxBVEVOVCIsImxpbmsiOjE1fSx7Im5hbWUi"
    "OiJ2YWUiLCJ0eXBlIjoiVkFFIiwibGluayI6MTZ9XSwib3V0cHV0cyI6W3sibmFtZSI6IkFVRElPIiwidHlwZSI6IkFVRElPIiwi"
    "bGlua3MiOlsxOF19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlZBRURlY29kZUF1ZGlvIn0sIndpZGdldHNf"
    "dmFsdWVzIjpbXX0seyJpZCI6MTUsInR5cGUiOiJDcmVhdGVWaWRlbyIsInBvcyI6WzI3NjAsMF0sInNpemUiOls0MDAsMTYwXSwi"
    "ZmxhZ3MiOnt9LCJvcmRlciI6MTUsIm1vZGUiOjAsImlucHV0cyI6W3sibmFtZSI6ImltYWdlcyIsInR5cGUiOiJJTUFHRSIsImxp"
    "bmsiOjE3fSx7Im5hbWUiOiJhdWRpbyIsInR5cGUiOiJBVURJTyIsImxpbmsiOjE4LCJzaGFwZSI6N31dLCJvdXRwdXRzIjpbeyJu"
    "YW1lIjoiVklERU8iLCJ0eXBlIjoiVklERU8iLCJsaW5rcyI6WzE5XX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZS"
    "IjoiQ3JlYXRlVmlkZW8ifSwid2lkZ2V0c192YWx1ZXMiOlsyNC4wLDgsInNSR0IiXX0seyJpZCI6MTYsInR5cGUiOiJTYXZlVmlk"
    "ZW8iLCJwb3MiOlszMjIwLDBdLCJzaXplIjpbNDAwLDEzNF0sImZsYWdzIjp7fSwib3JkZXIiOjE2LCJtb2RlIjowLCJpbnB1dHMi"
    "Olt7Im5hbWUiOiJ2aWRlbyIsInR5cGUiOiJWSURFTyIsImxpbmsiOjE5fV0sIm91dHB1dHMiOlt7Im5hbWUiOiJ2aWRlbyIsInR5"
    "cGUiOiJWSURFTyIsImxpbmtzIjpudWxsfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJTYXZlVmlkZW8ifSwi"
    "d2lkZ2V0c192YWx1ZXMiOlsiRmFzdEgzL2Zhc3RoM192Ml9uZXd2YWVfcjJ2IiwiYXV0byIsImF1dG8iXX0seyJpZCI6MTcsInR5"
    "cGUiOiJMb2FkSW1hZ2UiLCJwb3MiOlswLDE4MjBdLCJzaXplIjpbNDAwLDYwXSwiZmxhZ3MiOnt9LCJvcmRlciI6NywibW9kZSI6"
    "MCwiaW5wdXRzIjpbXSwib3V0cHV0cyI6W3sibmFtZSI6IklNQUdFIiwidHlwZSI6IklNQUdFIiwibGlua3MiOlsyMF19LHsibmFt"
    "ZSI6Ik1BU0siLCJ0eXBlIjoiTUFTSyIsImxpbmtzIjpudWxsfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJM"
    "b2FkSW1hZ2UifSwid2lkZ2V0c192YWx1ZXMiOlsiY2hhcnNoZWV0X2FuaW1lX2hvb2RpZV9ib3kucG5nIl19XSwibGlua3MiOltb"
    "MSwxLDAsMiwwLCJNT0RFTCJdLFsyLDIsMCwzLDAsIk1PREVMIl0sWzMsNCwwLDcsMCwiQ0xJUCJdLFs0LDUsMCw3LDEsIlZBRSJd"
    "LFs1LDYsMCw3LDIsIlZBRSJdLFs2LDMsMCw5LDAsIk1PREVMIl0sWzcsNywwLDksMSwiQ09ORElUSU9OSU5HIl0sWzgsOCwwLDEy"
    "LDAsIk5PSVNFIl0sWzksOSwwLDEyLDEsIkdVSURFUiJdLFsxMCwxMCwwLDEyLDIsIlNBTVBMRVIiXSxbMTEsMTEsMCwxMiwzLCJT"
    "SUdNQVMiXSxbMTIsNywxLDEyLDQsIkxBVEVOVCJdLFsxMywxMiwwLDEzLDAsIkxBVEVOVCJdLFsxNCw1LDAsMTMsMSwiVkFFIl0s"
    "WzE1LDEyLDAsMTQsMCwiTEFURU5UIl0sWzE2LDYsMCwxNCwxLCJWQUUiXSxbMTcsMTMsMCwxNSwwLCJJTUFHRSJdLFsxOCwxNCww"
    "LDE1LDEsIkFVRElPIl0sWzE5LDE1LDAsMTYsMCwiVklERU8iXSxbMjAsMTcsMCw3LDMsIklNQUdFIl1dLCJncm91cHMiOltdLCJj"
    "b25maWciOnt9LCJleHRyYSI6eyJkcyI6eyJzY2FsZSI6MC43LCJvZmZzZXQiOlswLDBdfX0sInZlcnNpb24iOjAuNH0="
)

_WF_R2V_APP_B64 = (
    "..."
    "ZWFyTW9kZSI6dHJ1ZX0sInZlcnNpb24iOjAuNH0="
)

_FASTH3_WFS = {"fasth3_v2_newvae_t2v.json": _WF_T2V_B64,
               "fasth3_v2_newvae_i2v.json": _WF_I2V_B64,
               "fasth3_v2_newvae_r2v.json": _WF_R2V_B64,
               "fasth3_v2_newvae_r2v_app.json": _WF_R2V_APP_B64}

print("")
print("📥 FastH3のワークフローを配置します（UI形式・画面から開けます）...")
for _n, _b in _FASTH3_WFS.items():
    _p = _os.path.join(h3_workflow_dir, _n)
    with open(_p, "w", encoding="utf-8") as _f:
        _f.write(_b64.b64decode(_b).decode("utf-8"))
    _wf = _json.load(open(_p, encoding="utf-8"))
    print("  ✓ %s（ノード%d / 接続%d）" % (_n, len(_wf["nodes"]), len(_wf["links"])))


print("""
FastH3 V2 + INT8映像VAE
・8ステップ、動画shift 10、音声shift 3、VSA保持率20%の組み合わせです。
・Workflows > User workflows > FastH3_V2_newVAE から選んでください。
・まず低解像度・短尺で確認してください。Colab L4/A100でのこの版の生成実測は未実施です。
・RTX 4090の別環境では1344×768、8秒、参照7枚で生成成功しています。
・R2V/I2Vは実験的です。参照の一致や終端の構図を保証しません。
・参照画像はLoadImageからReferenceToVideoへ線がつながっていることを確認してください。
・ログの BlockSparseAttention: sparse producer path / VSA tiles が高速化の確認点です。
""")

# ========================================
# 🆕 サンプル画像の自動ダウンロード（inputフォルダへ格納）
# ========================================
print("\n📥 Downloading sample image to input folder...")
input_dir = "/content/ComfyUI/input"

# 参照画像・起点画像のサンプル（すべて架空のキャラクター／人物です）
model_download("https://raw.githubusercontent.com/zasuko/zasuko-fasth3-colab/main/samples/charsheet_anime_hoodie_boy.png", input_dir)   # アニメ調のキャラクターシート（R2V向け）
model_download("https://raw.githubusercontent.com/zasuko/zasuko-fasth3-colab/main/samples/charsheet_anime_vtuber_rion.png", input_dir)   # アニメ調・配色見本つき
model_download("https://raw.githubusercontent.com/zasuko/zasuko-fasth3-colab/main/samples/charsheet_anime_maid_girl.png", input_dir)   # アニメ調
model_download("https://raw.githubusercontent.com/zasuko/zasuko-fasth3-colab/main/samples/charsheet_photo_mascot_cat.png", input_dir)   # 実写調・着ぐるみ
model_download("https://raw.githubusercontent.com/zasuko/zasuko-fasth3-colab/main/samples/charsheet_photo_man_glasses.png", input_dir)   # 実写調
model_download("https://raw.githubusercontent.com/zasuko/zasuko-fasth3-colab/main/samples/charsheet_photo_woman_pink.png", input_dir)   # 実写調
model_download("https://raw.githubusercontent.com/zasuko/zasuko-fasth3-colab/main/samples/portrait_photo_man_a.png", input_dir)   # バストアップ（I2Vの起点向け）
model_download("https://raw.githubusercontent.com/zasuko/zasuko-fasth3-colab/main/samples/portrait_photo_man_b.png", input_dir)   # バストアップ（I2Vの起点向け）

print("✓ All sample images downloaded!\n")



# ========================================
# ComfyUI起動設定
# ========================================

# 🆕 出力ディレクトリの設定
output_dir_arg = ""
if use_google_drive and enable_gdrive_output:
    output_dir_arg = f"--output-directory {GDRIVE_OUTPUT}"
    print("=" * 70)
    print(f"✅ ComfyUI output will be saved directly to Google Drive")
    print(f"📁 Output path: {GDRIVE_OUTPUT}")
    print("=" * 70)

print("🚀 Pinggy Tunnel を準備しています...")
import subprocess
import threading
import time
import socket
import urllib.request
import html
import re
from IPython.display import HTML, display

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\nComfyUI finished loading... setting up Pinggy tunnel!\n")
    print("(Pinggyは、Colab上のComfyUIを外部から使えるようにする一時トンネルサービスです。下に出るURLはこのセッション限りの一時URLです)\n")

    # ★ Pinggy有料トークンをColabシークレットから取得
    # Colab左側の「シークレット」に登録している名前に合わせています。
    # バックグラウンドスレッドからのシークレット取得はタイムアウトしやすいため、リトライする。
    from google.colab import userdata
    PINGGY_TOKEN = None
    for _pinggy_attempt in range(5):
        try:
            PINGGY_TOKEN = userdata.get("PINGGY_TOKEN")
            break
        except Exception as _pinggy_err:
            print(f"⚠️ PINGGY_TOKENの取得に失敗（{_pinggy_attempt + 1}/5回目）: {_pinggy_err}")
            time.sleep(3)

    if not PINGGY_TOKEN:
        raise ValueError(
            "Colabのシークレットに PINGGY_TOKEN が登録されていないか、5回リトライしても取得できませんでした。\n"
            "左側の🔑シークレットで名前とノートブックからのアクセス権（ON）を確認し、\n"
            "改善しない場合はこのセルを一度停止してから再実行してください。"
        )

    # ★ 有料Pinggy用に変更
    # 変更前: a.pinggy.io
    # 変更後: <Pinggyトークン>@pro.pinggy.io
    p = subprocess.Popen(["ssh", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=60", "-p", "443", "-R0:localhost:{}".format(port), f"{PINGGY_TOKEN}@pro.pinggy.io"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stdout:
        l = line.decode()
        match = re.search(r"https://[^\s]+", l)
        if match and "dashboard.pinggy.io" not in match.group(0):
            comfyui_url = match.group(0).rstrip(".,;)]}")
            print("\n" + "="*70)
            print("🚀 Pinggy URL:", comfyui_url)
            print("="*70 + "\n")
            safe_url = html.escape(comfyui_url, quote=True)
            display(HTML(
                f'<a href="{safe_url}" target="_blank" '
                'style="display:inline-block;padding:12px 20px;background:#1976d2;color:white;'
                'font-weight:bold;text-decoration:none;border-radius:8px">ComfyUIを開く</a>'
            ))
            print("✅URLからリンクを開いたら Pinggy の赤色の🟥「Enter Site」のボタンをクリックして下さい。")
            print("有料トークンで接続しているため、無料枠の60分制限ではなくPinggy Proの設定で動作します。")
            break

clear_output()
threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

# ★ 最重要修正箇所: --listen 0.0.0.0 と --enable-cors-header を追加して 403 Forbidden を回避します
# @markdown ---
# @markdown ## ⚡ Sage Attention（上級者向け、既定OFF推奨）
# @markdown 🌱FastH3はVSA(Sol-Attn)で高速化しているため、Sage Attentionは不要です。**必ずOFFのままにしてください**（併用すると衝突します）。
use_sage_attention = False  # @param {type:"boolean"}

if use_sage_attention:
    raise RuntimeError("このV2版ではSage AttentionをOFFにしてください。")
_sage_flag = ""

if using_L4_GPU:
    command = f"python main.py {output_dir_arg} --listen 0.0.0.0 --enable-cors-header --cache-none{_sage_flag} --dont-print-server".strip()
    get_ipython().system(command)
else:
    command = f"python main.py {output_dir_arg} --listen 0.0.0.0 --enable-cors-header{_sage_flag} --dont-print-server".strip()
    get_ipython().system(command)



# モデル・設定の出所

| 部品 | この版の設定 |
|---|---|
| メインモデル | `fastvideo_fasth3_8step_v2_pruned_int8_convrot.safetensors` |
| 映像VAE | `minimax_h3_video_vae_int8_convrot.safetensors` |
| 文章・参照の理解 | `qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors` |
| 音声VAE | `minimax_h3_audio_vae_fp32.safetensors` |
| ComfyUI | v0.36.0 / `ee71d5c4993f29086b27fde1629a945ae48425bf` |
| PyTorch | 2.9.0 / CUDA 13.0（元Notebookの組み合わせを維持） |
| 高速化 | comfy-kitchen 0.2.34、標準BlockSparseAttention、VSA保持率20% |
| ManualSigmas | `0.9998999099, 0.9857884051, 0.9675752487, 0.9431680774, 0.9090909091, 0.8571428571, 0.7692307692, 0.5882352941, 0` |

- [V2公式モデルカード](https://huggingface.co/FastVideo/FastVideo-FastH3-8-Step-V2)
- [ComfyUI用V2配布](https://huggingface.co/FastVideo/FastVideo-FastH3-Comfy)
- [VAE・テキストエンコーダー配布](https://huggingface.co/Comfy-Org/MiniMax-H3)
- [FastVideo公式設定](https://haoailab.com/FastVideo/inference/fasth3-distilled/)
- [ComfyUI v0.36.0](https://github.com/Comfy-Org/ComfyUI/tree/ee71d5c4993f29086b27fde1629a945ae48425bf)
- [モデルのMiniMax H3 Community License](https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/LICENSE)

モデルは配布元から直接ダウンロードし、このGitHubリポジトリには再配布しません。利用地域・用途など、モデル配布元の条件を確認してください。ComfyUIはGPL-3.0、comfy-kitchenはApache-2.0です。旧版の専用wheelはこの追加版では使いません。

2026-09-20：Notebook形式・Python構文・全ワークフローの接続とV2設定をローカルで確認。Colabでのインストール・トンネル接続・生成の実機試験は未実施です。
